# Mission 3 — 90/10 층화 실험 → 모델 선택 → Training 100% 최종 재학습

이 노트북은 검증된 **MISSION3_FINAL_TRAIN100_VALIDATE** 실행 구조를 바탕으로 아래 실험을 한 번에 수행합니다.

1. Training 90/10 multilabel stratified 고정 분할
2. 오심 hard-negative 제외
3. 전신쇠약·호흡곤란 hard-negative 1.5배 노출
4. 모든 원본 데이터를 매 epoch 최소 1회 보장하는 sampler
5. 현재 입력과 화자 역할 토큰 입력 비교
6. 내부 검증의 고정 0.5 Macro F1로 best epoch 선택
7. 허용 시 내부 검증에서만 증상별 임계값 선택
8. 최종 설정으로 Training 100% 재학습 후 공식 Validation 1회 평가

Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU 이상**을 선택한 뒤 **런타임 → 모두 실행**하세요. 중단되면 같은 노트북에서 다시 모두 실행하면 완료된 단계를 재사용합니다.


## 1. 패키지 준비

Colab의 CUDA용 PyTorch는 유지하고 필요한 라이브러리만 설치합니다.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-compile",
    "transformers>=4.36,<6", "pandas>=2.0", "tqdm>=4.66",
])
print("패키지 준비 완료")

## 2. Google Drive 연결

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import time

DRIVE_MOUNT = Path("/content/drive")

def mount_drive_with_recovery(force=False):
    try:
        drive.mount(str(DRIVE_MOUNT), force_remount=force)
        # '이미 마운트됨' 메시지만 믿지 않고 실제 읽기가 되는지 확인합니다.
        next((DRIVE_MOUNT / "MyDrive").iterdir(), None)
    except OSError as error:
        if error.errno != 107:
            raise
        print("Drive 연결이 끊겨 마운트를 복구합니다.")
        subprocess.run(
            ["fusermount", "-u", str(DRIVE_MOUNT)],
            check=False,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(2)
        drive.mount(str(DRIVE_MOUNT), force_remount=True)
        next((DRIVE_MOUNT / "MyDrive").iterdir(), None)

mount_drive_with_recovery(force=False)
print("Google Drive 읽기 확인 완료")

## 3. 경로와 실험 설정

기존 최종학습 노트북의 Drive 경로를 우선 사용합니다. `대학부 데이터`가 Drive 바로가기라 일반 폴더처럼 보이지 않는 경우에는 Colab이 마운트한 바로가기 대상만 추가로 확인합니다.

아래 셀에서 보통 수정할 값은 없습니다. 자동 확인에 실패했을 때만 `MANUAL_DATA_ROOT` 또는 `MANUAL_BASELINE_CHECKPOINT`에 실제 절대경로를 넣으세요.


In [ ]:
from pathlib import Path
import os

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/MISSION3_TURN_CHUNK")
LOCAL_PROJECT_DIR = Path("/content/MISSION3_TURN_CHUNK")

# 자동 확인 실패 때만 문자열로 지정합니다. 예: "/content/drive/MyDrive/DCC_MISSION3_100PCT/Data"
MANUAL_DATA_ROOT = None
MANUAL_BASELINE_CHECKPOINT = None

RUN_NAME = "stratified_role_compare_seed42_v2"
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/DCC_MISSION3_100PCT") / RUN_NAME

MODEL_NAME = "klue/roberta-large"
SEED = 42
INTERNAL_VAL_RATIO = 0.10
MAX_EPOCHS = 5
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 1e-5
MAX_LENGTH = 256
CHUNK_OVERLAP = 64
POS_WEIGHT_SCALE = 0.25
POS_WEIGHT_MAX = 3.0
HARD_NEGATIVE_OVERSAMPLE = 1.5
HARD_NEGATIVE_SYMPTOMS = ["전신쇠약", "호흡곤란"]
ALLOW_INTERNAL_THRESHOLD_TUNING = True


def _dataset_layout(root):
    """Training TL_*와 Validation VL_*가 모두 있는 데이터 루트인지 확인합니다."""
    if root is None:
        return None
    root = Path(root)
    try:
        train_dirs = []
        val_dirs = []
        for folder in ("Training", "training"):
            train_dirs.extend(root.glob(folder + "/2.*/TL_*"))
        for folder in ("Validation", "validation"):
            val_dirs.extend(root.glob(folder + "/2.*/VL_*"))
        if train_dirs and val_dirs:
            return root
    except OSError:
        return None
    return None


def resolve_data_root():
    candidates = []
    if MANUAL_DATA_ROOT:
        candidates.append(Path(MANUAL_DATA_ROOT))
    candidates.extend([
        Path("/content/drive/MyDrive/대학부 데이터"),
        Path("/content/drive/MyDrive/DCC_MISSION3_100PCT/Data"),
        Path("/content/drive/MyDrive/DCC_MISSION3_100PCT/데이터"),
        Path("/content/drive/MyDrive/DCC_MISSION3/Data"),
        Path("/content/drive/MyDrive/DCC_MISSION3/데이터"),
        Path("/content/drive/MyDrive/DCC_2026/Data"),
        Path("/content/drive/MyDrive/DCC_2026/데이터"),
        DRIVE_PROJECT_DIR / "Data",
    ])

    checked = []
    for candidate in candidates:
        checked.append(str(candidate))
        resolved = _dataset_layout(candidate)
        if resolved is not None:
            return resolved

    # Google Drive 바로가기는 MyDrive 아래에서 is_dir()가 False일 수 있습니다.
    shortcut_root = Path("/content/drive/.shortcut-targets-by-id")
    if shortcut_root.exists():
        try:
            label_dirs = list(shortcut_root.glob("*/Training/2.*/TL_*"))
            label_dirs += list(shortcut_root.glob("*/*/Training/2.*/TL_*"))
            if not label_dirs:
                label_dirs = list(shortcut_root.glob("**/Training/2.*/TL_*"))
            for label_dir in label_dirs:
                root = label_dir.parents[2]
                checked.append(str(root))
                resolved = _dataset_layout(root)
                if resolved is not None:
                    return resolved
        except OSError:
            pass

    raise FileNotFoundError(
        "Training/2.*/TL_*와 Validation/2.*/VL_*가 함께 있는 데이터 루트를 찾지 못했습니다.\n"
        "확인한 후보:\n- " + "\n- ".join(checked) + "\n\n"
        "Drive의 '대학부 데이터'가 바로가기인데 Colab이 대상을 노출하지 않는 경우, "
        "바로가기 대상 폴더의 실제 경로를 MANUAL_DATA_ROOT에 넣거나 "
        "DCC_MISSION3_100PCT/Data에 데이터 폴더를 복사하세요."
    )


def resolve_baseline_checkpoint():
    candidates = []
    if MANUAL_BASELINE_CHECKPOINT:
        candidates.append(Path(MANUAL_BASELINE_CHECKPOINT))
    candidates.extend([
        Path("/content/drive/MyDrive/DCC_MISSION3/hardneg_klue_roberta_large_train100_seed42/final.pt"),
        DRIVE_PROJECT_DIR / "checkpoints" / "mission3_kcbert_turn_chunked_40pct_weight025_effbatch16_3epochs.pt",
        Path("/content/drive/MyDrive/DCC_MISSION3_100PCT/final.pt"),
    ])
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "기준 체크포인트를 찾지 못했습니다. 확인한 후보:\n- "
        + "\n- ".join(str(path) for path in candidates)
        + "\n실제 final.pt 경로를 MANUAL_BASELINE_CHECKPOINT에 넣으세요."
    )


mount_drive_with_recovery(force=False)
DATA_ROOT = resolve_data_root()
BASELINE_CHECKPOINT = resolve_baseline_checkpoint()
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CHECKPOINT = DRIVE_RUN_DIR / "final" / "final.pt"
EXPERIMENT_SUMMARY = DRIVE_RUN_DIR / "experiment_summary.json"
OFFICIAL_METRICS = DRIVE_RUN_DIR / "official_validation_metrics.json"
FINAL_VERIFICATION = DRIVE_RUN_DIR / "FINAL_VERIFICATION.json"

if FINAL_CHECKPOINT.is_file() and EXPERIMENT_SUMMARY.is_file():
    RUN_MODE = "evaluate_only"
elif any(DRIVE_RUN_DIR.iterdir()):
    RUN_MODE = "resume"
else:
    RUN_MODE = "fresh"

print("DATA_ROOT:", DATA_ROOT)
print("BASELINE_CHECKPOINT:", BASELINE_CHECKPOINT)
print("DRIVE_RUN_DIR:", DRIVE_RUN_DIR)
print("RUN_MODE:", RUN_MODE)


## 4. GPU·경로 확인 및 내장 코드 준비

실행에 필요한 Python 파일을 노트북 안에 포함했습니다. Drive의 오래된 `.py` 파일을 사용하지 않으므로 파일 버전이 섞이지 않습니다. 각 파일은 SHA-256으로 확인한 뒤 `/content/MISSION3_TURN_CHUNK`에 복원합니다.


In [ ]:
import base64
import hashlib
import json
import torch

mount_drive_with_recovery(force=False)

BUNDLE_VERSION = "stratified-role-train100-v2-20261007"
EMBEDDED_FILES = {"config.py":"IiIiTWlzc2lvbiAzIEtjQkVSVCDsoITssrQg7LKt7YGsIOuqqOuNuCDshKTsoJUuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgZ2xvYgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKCiMg64uk66W4IFBD7JeQ7ISc64qUIO2ZmOqyvSDrs4DsiJggRENDX0RBVEFfUk9PVOulvCDsp4DsoJXtlZjrqbQg7IaM7IqkIOyImOyglSDsl4bsnbQg7IKs7Jqp7ZWgIOyImCDsnojri6QuCkRBVEFfUk9PVCA9IFBhdGgoCiAgICBvcy5lbnZpcm9uLmdldCgKICAgICAgICAiRENDX0RBVEFfUk9PVCIsCiAgICAgICAgciJDOlxVc2Vyc1zsobDsp4DsnYBcRGVza3RvcFxBam91XERDQ1zsmIjshKDrjbDsnbTthLBcRGF0YSIsCiAgICApCikKCkNIRUNLUE9JTlRfRElSID0gUk9PVCAvICJjaGVja3BvaW50cyIKT1VUUFVUX0RJUiA9IFJPT1QgLyAib3V0cHV0cyIKQ0hVTktfU1RSQVRFR1kgPSAiZGlhbG9ndWVfdHVybl9vdmVybGFwX3YxIgpERUZBVUxUX0NIRUNLUE9JTlRfUEFUSCA9IENIRUNLUE9JTlRfRElSIC8gIm1pc3Npb24zX2tjYmVydF90dXJuX2NodW5rZWQucHQiCkRFRkFVTFRfU1VCTUlTU0lPTl9QQVRIID0gT1VUUFVUX0RJUiAvICJtaXNzaW9uM19rY2JlcnRfdHVybl9jaHVua2VkLmNzdiIKClRBUkdFVF9TWU1QVE9NUyA9IFsKICAgICLqs6Dsl7QiLAogICAgIuq1rO2GoCIsCiAgICAi65GQ7Ya1IiwKICAgICLrs7XthrUiLAogICAgIuyWtOyngOufrOybgCIsCiAgICAi7Je07IOBIiwKICAgICLsmKTsi6wiLAogICAgIuyghOyLoOyHoOyVvSIsCiAgICAi7Zi47Z2h6rOk656AIiwKXQoKCmRlZiBfcmVzb2x2ZV9kYXRhX2RpcigKICAgIHBhdHRlcm5zOiBzdHIgfCB0dXBsZVtzdHIsIC4uLl0sIGRhdGFfcm9vdDogUGF0aCB8IE5vbmUgPSBOb25lCikgLT4gUGF0aDoKICAgICIiIuuMgOyGjOusuOyekCDtkZzquLDqsIAg64uk66W4IOuNsOydtO2EsCDtj7TrjZQg7ZuE67O0IOykkSDssqsg6rK966Gc66W8IOuwmO2ZmC4iIiIKICAgIHJvb3QgPSBQYXRoKGRhdGFfcm9vdCkgaWYgZGF0YV9yb290IGlzIG5vdCBOb25lIGVsc2UgREFUQV9ST09UCiAgICBjYW5kaWRhdGVzID0gKHBhdHRlcm5zLCkgaWYgaXNpbnN0YW5jZShwYXR0ZXJucywgc3RyKSBlbHNlIHBhdHRlcm5zCiAgICBmb3IgcGF0dGVybiBpbiBjYW5kaWRhdGVzOgogICAgICAgIG1hdGNoZXMgPSBzb3J0ZWQoZ2xvYi5nbG9iKHN0cihyb290IC8gcGF0dGVybikpKQogICAgICAgIGlmIG1hdGNoZXM6CiAgICAgICAgICAgIHJldHVybiBQYXRoKG1hdGNoZXNbMF0pCiAgICBhdHRlbXB0ZWQgPSAiLCAiLmpvaW4oc3RyKHJvb3QgLyBwYXR0ZXJuKSBmb3IgcGF0dGVybiBpbiBjYW5kaWRhdGVzKQogICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiLrjbDsnbTthLAg7Y+0642UIOyXhuydjC4g7ZmV7J247ZWcIOqyveuhnDoge2F0dGVtcHRlZH0iKQoKCmRlZiBnZXRfdHJhaW5fbGFiZWxfZGlyKGRhdGFfcm9vdDogUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBQYXRoOgogICAgcmV0dXJuIF9yZXNvbHZlX2RhdGFfZGlyKCgiVHJhaW5pbmcvMi4qL1RMXyoiLCAidHJhaW5pbmcvMi4qL1RMXyoiKSwgZGF0YV9yb290KQoKCmRlZiBnZXRfdmFsX2xhYmVsX2RpcihkYXRhX3Jvb3Q6IFBhdGggfCBOb25lID0gTm9uZSkgLT4gUGF0aDoKICAgIHJldHVybiBfcmVzb2x2ZV9kYXRhX2RpcigoIlZhbGlkYXRpb24vMi4qL1ZMXyoiLCAidmFsaWRhdGlvbi8yLiovVkxfKiIpLCBkYXRhX3Jvb3QpCg==","data_utils.py":"IiIiTWlzc2lvbiAzIOudvOuyqCBKU09OIO2MjOyLsSDrsI8g7YWN7Iqk7Yq4L+ymneyDgSDrs4DtmZgg7Jyg7Yu466as7YuwLiIiIg0KDQppbXBvcnQganNvbg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55DQoNCmltcG9ydCB0b3JjaA0KDQpmcm9tIGNvbmZpZyBpbXBvcnQgVEFSR0VUX1NZTVBUT01TDQpmcm9tIGRpYWxvZ3VlX2NodW5raW5nIGltcG9ydCBidWlsZF9kaWFsb2d1ZV90dXJucywgYnVpbGRfZGlhbG9ndWVfdG9rZW5fY2h1bmtzDQoNCg0KZGVmIGxvYWRfbGFiZWwobGFiZWxfcGF0aDogUGF0aCkgLT4gZGljdFtzdHIsIEFueV06DQogICAgIiIiSlNPTiDrnbzrsqgg7YyM7J287J2EIFVURi0466GcIOuhnOuTnC4iIiINCiAgICB3aXRoIG9wZW4obGFiZWxfcGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmlsZToNCiAgICAgICAgcmV0dXJuIGpzb24ubG9hZChmaWxlKQ0KDQoNCmRlZiBidWlsZF9kaWFsb2d1ZV90ZXh0KGxhYmVsOiBkaWN0W3N0ciwgQW55XSkgLT4gc3RyOg0KICAgICIiIuuqqOuToCB1dHRlcmFuY2XsnZgg7KCE7IKsIO2FjeyKpO2KuOulvCDsiJzshJzrjIDroZwg6rO167CxIOyXsOqysC4iIiINCiAgICB0ZXh0cyA9IFt1LmdldCgidGV4dCIsICIiKS5zdHJpcCgpIGZvciB1IGluIGxhYmVsLmdldCgidXR0ZXJhbmNlcyIsIFtdKV0NCiAgICByZXR1cm4gIiAiLmpvaW4odGV4dCBmb3IgdGV4dCBpbiB0ZXh0cyBpZiB0ZXh0KQ0KDQoNCmRlZiB0b2tlbml6ZV90ZXh0X2NodW5rcygNCiAgICB0b2tlbml6ZXIsDQogICAgdGV4dDogc3RyLA0KICAgIG1heF9sZW5ndGg6IGludCA9IDI1NiwNCiAgICBjaHVua19vdmVybGFwOiBpbnQgPSA2NCwNCikgLT4gZGljdFtzdHIsIHRvcmNoLlRlbnNvcl06DQogICAgIiIi7YWN7Iqk7Yq4IOyghOyytOulvCDqsrnsuZjripQg7Yag7YGwIOyyre2BrOuhnCDrgpjriKAgcGFkZGluZ+2VnCDthZDshJzroZwg67CY7ZmYLg0KDQogICAg7Yq57IiYIO2GoO2BsChbQ0xTXSwgW1NFUF0p7J20IOuTpOyWtOqwiCDsnpDrpqzrpbwg66i87KCAIOygnOyZuO2VmOuvgOuhnCDqsIEg7LKt7YGs64qUDQogICAgYGBtYXhfbGVuZ3RoYGDrpbwg64SY7KeAIOyViuuKlOuLpC4g66eI7KeA66eJIOyyre2BrOq5jOyngCDsg53shLHtlZjrr4DroZwg7JuQ66y47J2YIOyWtOuWpA0KICAgIO2GoO2BsOuPhCB0cnVuY2F0aW9u7Jy866GcIOuyhOumrOyngCDslYrripTri6QuDQogICAgIiIiDQogICAgc3BlY2lhbF90b2tlbnMgPSB0b2tlbml6ZXIubnVtX3NwZWNpYWxfdG9rZW5zX3RvX2FkZChwYWlyPUZhbHNlKQ0KICAgIGNvbnRlbnRfbGVuZ3RoID0gbWF4X2xlbmd0aCAtIHNwZWNpYWxfdG9rZW5zDQogICAgaWYgY29udGVudF9sZW5ndGggPD0gMDoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigNCiAgICAgICAgICAgIGYibWF4X2xlbmd0aD17bWF4X2xlbmd0aH3qsIAg7Yq57IiYIO2GoO2BsCDsiJgge3NwZWNpYWxfdG9rZW5zfeuztOuLpCDsu6Tslbwg7ZWp64uI64ukLiINCiAgICAgICAgKQ0KICAgIGlmIG5vdCAwIDw9IGNodW5rX292ZXJsYXAgPCBjb250ZW50X2xlbmd0aDoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigNCiAgICAgICAgICAgICJjaHVua19vdmVybGFw7J2AIDAg7J207IOB7J2066mwIOyyre2BrOydmCDrs7jrrLgg7Yag7YGwIOyImOuztOuLpCDsnpHslYTslbwg7ZWp64uI64ukLiAiDQogICAgICAgICAgICBmIijtmITsnqwgY2h1bmtfb3ZlcmxhcD17Y2h1bmtfb3ZlcmxhcH0sIOuzuOusuCDthqDtgbAg7IiYPXtjb250ZW50X2xlbmd0aH0pIg0KICAgICAgICApDQoNCiAgICB0b2tlbl9pZHMgPSB0b2tlbml6ZXIoDQogICAgICAgIHRleHQsDQogICAgICAgIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSwNCiAgICAgICAgdHJ1bmNhdGlvbj1GYWxzZSwNCiAgICAgICAgcmV0dXJuX2F0dGVudGlvbl9tYXNrPUZhbHNlLA0KICAgICAgICB2ZXJib3NlPUZhbHNlLA0KICAgIClbImlucHV0X2lkcyJdDQogICAgc3RlcCA9IGNvbnRlbnRfbGVuZ3RoIC0gY2h1bmtfb3ZlcmxhcA0KICAgIHJhd19jaHVua3M6IGxpc3RbbGlzdFtpbnRdXSA9IFtdDQoNCiAgICBpZiBub3QgdG9rZW5faWRzOg0KICAgICAgICByYXdfY2h1bmtzLmFwcGVuZChbXSkNCiAgICBlbHNlOg0KICAgICAgICBzdGFydCA9IDANCiAgICAgICAgd2hpbGUgc3RhcnQgPCBsZW4odG9rZW5faWRzKToNCiAgICAgICAgICAgIHJhd19jaHVua3MuYXBwZW5kKHRva2VuX2lkc1tzdGFydCA6IHN0YXJ0ICsgY29udGVudF9sZW5ndGhdKQ0KICAgICAgICAgICAgaWYgc3RhcnQgKyBjb250ZW50X2xlbmd0aCA+PSBsZW4odG9rZW5faWRzKToNCiAgICAgICAgICAgICAgICBicmVhaw0KICAgICAgICAgICAgc3RhcnQgKz0gc3RlcA0KDQogICAgcmV0dXJuIF9wYWRfdG9rZW5fY2h1bmtzKHRva2VuaXplciwgcmF3X2NodW5rcywgbWF4X2xlbmd0aCkNCg0KDQpkZWYgX3BhZF90b2tlbl9jaHVua3ModG9rZW5pemVyLCByYXdfY2h1bmtzLCBtYXhfbGVuZ3RoKToNCiAgICAiIiLrs7jrrLgg7Yag7YGwIOyyre2BrOyXkCDtirnsiJgg7Yag7YGw7J2EIOy2lOqwgO2VmOqzoCDqsJnsnYAg6ri47J2066GcIHBhZGRpbmcuIiIiDQogICAgIyB0cmFuc2Zvcm1lcnMgNS547J2YIOydvOu2gCBzbG93IEJlcnRUb2tlbml6ZXLsl5DshJzripQg7JiI7KCEIOqzteqwnCDrqZTshJzrk5zsnbgNCiAgICAjIGJ1aWxkX2lucHV0c193aXRoX3NwZWNpYWxfdG9rZW5z6rCAIOygnOqzteuQmOyngCDslYrripTri6QuIEtjQkVSVOuKlCDri6jsnbwg66y47J6l7J2EDQogICAgIyBbQ0xTXSDrs7jrrLggW1NFUF0g7Iic7ISc66GcIOq1rOyEse2VmOuvgOuhnCDtlbTri7kg67KE7KCE7JeQ7ISc64qUIO2GoO2BsCBJROulvCDsp4HsoJEg67aZ7J2464ukLg0KICAgIGJ1aWxkX3dpdGhfc3BlY2lhbHMgPSBnZXRhdHRyKA0KICAgICAgICB0b2tlbml6ZXIsDQogICAgICAgICJidWlsZF9pbnB1dHNfd2l0aF9zcGVjaWFsX3Rva2VucyIsDQogICAgICAgIE5vbmUsDQogICAgKQ0KICAgIGlmIGNhbGxhYmxlKGJ1aWxkX3dpdGhfc3BlY2lhbHMpOg0KICAgICAgICBjaHVua3Nfd2l0aF9zcGVjaWFscyA9IFtidWlsZF93aXRoX3NwZWNpYWxzKGNodW5rKSBmb3IgY2h1bmsgaW4gcmF3X2NodW5rc10NCiAgICBlbHNlOg0KICAgICAgICBjbHNfdG9rZW5faWQgPSB0b2tlbml6ZXIuY2xzX3Rva2VuX2lkDQogICAgICAgIHNlcF90b2tlbl9pZCA9IHRva2VuaXplci5zZXBfdG9rZW5faWQNCiAgICAgICAgaWYgY2xzX3Rva2VuX2lkIGlzIE5vbmUgb3Igc2VwX3Rva2VuX2lkIGlzIE5vbmU6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLthqDtgazrgpjsnbTsoIDsl5AgQ0xTL1NFUCDtirnsiJgg7Yag7YGwIElE6rCAIOyXhuyKteuLiOuLpC4iKQ0KICAgICAgICBjaHVua3Nfd2l0aF9zcGVjaWFscyA9IFsNCiAgICAgICAgICAgIFtjbHNfdG9rZW5faWQsICpjaHVuaywgc2VwX3Rva2VuX2lkXQ0KICAgICAgICAgICAgZm9yIGNodW5rIGluIHJhd19jaHVua3MNCiAgICAgICAgXQ0KICAgIGVuY29kZWQgPSB0b2tlbml6ZXIucGFkKA0KICAgICAgICB7ImlucHV0X2lkcyI6IGNodW5rc193aXRoX3NwZWNpYWxzfSwNCiAgICAgICAgcGFkZGluZz0ibWF4X2xlbmd0aCIsDQogICAgICAgIG1heF9sZW5ndGg9bWF4X2xlbmd0aCwNCiAgICAgICAgcmV0dXJuX2F0dGVudGlvbl9tYXNrPVRydWUsDQogICAgICAgIHJldHVybl90ZW5zb3JzPSJwdCIsDQogICAgKQ0KICAgIHJldHVybiB7DQogICAgICAgICJpbnB1dF9pZHMiOiBlbmNvZGVkWyJpbnB1dF9pZHMiXSwNCiAgICAgICAgImF0dGVudGlvbl9tYXNrIjogZW5jb2RlZFsiYXR0ZW50aW9uX21hc2siXSwNCiAgICB9DQoNCg0KZGVmIHRva2VuaXplX2RpYWxvZ3VlX2NodW5rcygNCiAgICB0b2tlbml6ZXIsDQogICAgdXR0ZXJhbmNlczogbGlzdFtkaWN0W3N0ciwgQW55XV0sDQogICAgbWF4X2xlbmd0aDogaW50ID0gMjU2LA0KICAgIGNodW5rX292ZXJsYXA6IGludCA9IDY0LA0KICAgIHNwZWFrZXJfcm9sZV90b2tlbnM6IGJvb2wgPSBGYWxzZSwNCikgLT4gZGljdFtzdHIsIHRvcmNoLlRlbnNvcl06DQogICAgIiIi67Cc7ZmUIOqyveqzhOulvCDqs6DroKTtlZwg7LKt7YGs66W8IOuqqOuNuCDsnoXroKUg7YWQ7ISc66GcIOuwmO2ZmC4iIiINCiAgICBjaHVua3MgPSBidWlsZF9kaWFsb2d1ZV90b2tlbl9jaHVua3MoDQogICAgICAgIHRva2VuaXplciwNCiAgICAgICAgdXR0ZXJhbmNlcywNCiAgICAgICAgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLA0KICAgICAgICBjaHVua19vdmVybGFwPWNodW5rX292ZXJsYXAsDQogICAgICAgIHNwZWFrZXJfcm9sZV90b2tlbnM9c3BlYWtlcl9yb2xlX3Rva2VucywNCiAgICApDQogICAgcmV0dXJuIF9wYWRfdG9rZW5fY2h1bmtzKHRva2VuaXplciwgY2h1bmtzLCBtYXhfbGVuZ3RoKQ0KDQoNCmRlZiBmaWx0ZXJfc3ltcHRvbXMoc3ltcHRvbXM6IGxpc3Rbc3RyXSkgLT4gbGlzdFtzdHJdOg0KICAgICIiIu2PieqwgCDrjIDsg4EgOeqwnCDspp3sg4Hrp4wg7Jyg7KeALiIiIg0KICAgIHRhcmdldF9zZXQgPSBzZXQoVEFSR0VUX1NZTVBUT01TKQ0KICAgIHJldHVybiBbc3ltcHRvbSBmb3Igc3ltcHRvbSBpbiBzeW1wdG9tcyBpZiBzeW1wdG9tIGluIHRhcmdldF9zZXRdDQoNCg0KZGVmIHN5bXB0b21zX3RvX211bHRpaG90KHN5bXB0b21zOiBsaXN0W3N0cl0pIC0+IGxpc3RbaW50XToNCiAgICAiIiLspp3sg4Eg66as7Iqk7Yq466W8IFRBUkdFVF9TWU1QVE9NUyDsiJzshJzsnZggOeywqOybkCBtdWx0aS1ob3TsnLzroZwg67OA7ZmYLiIiIg0KICAgIGZpbHRlcmVkID0gZmlsdGVyX3N5bXB0b21zKHN5bXB0b21zKQ0KICAgIHJldHVybiBbMSBpZiBzeW1wdG9tIGluIGZpbHRlcmVkIGVsc2UgMCBmb3Igc3ltcHRvbSBpbiBUQVJHRVRfU1lNUFRPTVNdDQoNCg0KZGVmIG11bHRpaG90X3RvX3N5bXB0b21fc3RyKG11bHRpaG90OiBsaXN0W2ludF0pIC0+IHN0cjoNCiAgICAiIiI57LCo7JuQIG11bHRpLWhvdOydhCBDU1Yg7Lac66Cl7JqpIOymneyDgSDrrLjsnpDsl7TroZwg67OA7ZmYLiIiIg0KICAgIHNlbGVjdGVkID0gWw0KICAgICAgICBUQVJHRVRfU1lNUFRPTVNbaW5kZXhdDQogICAgICAgIGZvciBpbmRleCwgdmFsdWUgaW4gZW51bWVyYXRlKG11bHRpaG90KQ0KICAgICAgICBpZiB2YWx1ZSA9PSAxDQogICAgXQ0KICAgIHJldHVybiBzdHIoc2VsZWN0ZWQpDQoNCg0KZGVmIGxpc3RfbGFiZWxfZmlsZXMobGFiZWxfZGlyOiBQYXRoKSAtPiBsaXN0W1BhdGhdOg0KICAgICIiIuudvOuyqCDrlJTroInthLDrpqzsnZggSlNPTiDtjIzsnbzsnYQg7YyM7J2866qF7Iic7Jy866GcIOuwmO2ZmC4iIiINCiAgICByZXR1cm4gc29ydGVkKGxhYmVsX2Rpci5nbG9iKCIqLmpzb24iKSkNCg==","dialogue_chunking.py":"IiIiUHlUb3JjaCDsl4bsnbQg7IKs7Jqp7ZWgIOyImCDsnojripQg67Cc7ZmUIOqyveqzhCDssq3tgawg7IOd7ISxIOuhnOyngS4iIiINCg0KZnJvbSB0eXBpbmcgaW1wb3J0IEFueQ0KDQoNCmRlZiBfc3BlYWtlcl9wcmVmaXgoc3BlYWtlcjogQW55KSAtPiBzdHI6DQogICAgIiIi642w7J207YSw7J2YIDAvMSDtmZTsnpAgSUTrpbwg66qo6424IOyeheugpeyaqSDsl63tlaAg7ZGc6riw66GcIOuzgO2ZmC4iIiINCiAgICBub3JtYWxpemVkID0gc3RyKHNwZWFrZXIpLnN0cmlwKCkubG93ZXIoKQ0KICAgIGlmIG5vcm1hbGl6ZWQgaW4geyIwIiwgIuyDgeuLtOybkCIsICJkaXNwYXRjaGVyIiwgImFnZW50In06DQogICAgICAgIHJldHVybiAiW+yDgeuLtOybkF0iDQogICAgaWYgbm9ybWFsaXplZCBpbiB7IjEiLCAi7Iug6rOg7J6QIiwgImNhbGxlciIsICJwYXRpZW50In06DQogICAgICAgIHJldHVybiAiW+yLoOqzoOyekF0iDQogICAgcmV0dXJuICJb7ZmU7J6QXSINCg0KDQpkZWYgYnVpbGRfZGlhbG9ndWVfdHVybnMoDQogICAgdXR0ZXJhbmNlczogbGlzdFtkaWN0W3N0ciwgQW55XV0sDQogICAgc3BlYWtlcl9yb2xlX3Rva2VuczogYm9vbCA9IEZhbHNlLA0KKSAtPiBsaXN0W3N0cl06DQogICAgIiIi7J6F66ClIOyInOyEnOulvCDsnKDsp4DtlZjrqbAg6rCZ7J2AIO2ZlOyekOydmCDsl7Dsho0g67Cc7ZmU66W8IO2VmOuCmOydmCDthLTsnLzroZwg66y264qU64ukLg0KDQogICAg7ZmU7J6QIOygleuztOqwgCDsl4bripQg67Cc7ZmU64qUIOuPheumvSDthLTsnLzroZwg7Jyg7KeA7ZWc64ukLiDtmZTsnpAg67KI7Zi47J2YIOyXre2VoOydtOuCmA0KICAgIOusvOydjO2RnOyXkCDsnZjsobTtlZjsp4Ag7JWK7Jy866+A66GcIOyduOygke2VnCDsp4jrrLjCt+uLteuzgCDrrLjrp6XsnYQg7ZWo6ruYIOuztOyhtO2VnOuLpC4NCiAgICAiIiINCiAgICB0dXJucyA9IFtdDQogICAgcHJldmlvdXNfc3BlYWtlciA9IE5vbmUNCiAgICBmb3IgdXR0ZXJhbmNlIGluIHV0dGVyYW5jZXM6DQogICAgICAgIHRleHQgPSB1dHRlcmFuY2UuZ2V0KCJ0ZXh0IiwgIiIpLnN0cmlwKCkNCiAgICAgICAgaWYgbm90IHRleHQ6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBzcGVha2VyID0gdXR0ZXJhbmNlLmdldCgic3BlYWtlciIpDQogICAgICAgIGlmIHR1cm5zIGFuZCBzcGVha2VyIGlzIG5vdCBOb25lIGFuZCBzcGVha2VyID09IHByZXZpb3VzX3NwZWFrZXI6DQogICAgICAgICAgICB0dXJuc1stMV0gKz0gIiAiICsgdGV4dA0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgcHJlZml4ID0gZiJ7X3NwZWFrZXJfcHJlZml4KHNwZWFrZXIpfSAiIGlmIHNwZWFrZXJfcm9sZV90b2tlbnMgZWxzZSAiIg0KICAgICAgICAgICAgdHVybnMuYXBwZW5kKHByZWZpeCArIHRleHQpDQogICAgICAgIHByZXZpb3VzX3NwZWFrZXIgPSBzcGVha2VyDQogICAgcmV0dXJuIHR1cm5zDQoNCmRlZiBidWlsZF9kaWFsb2d1ZV90b2tlbl9jaHVua3MoDQogICAgdG9rZW5pemVyLA0KICAgIHV0dGVyYW5jZXM6IGxpc3RbZGljdFtzdHIsIEFueV1dLA0KICAgIG1heF9sZW5ndGg6IGludCA9IDI1NiwNCiAgICBjaHVua19vdmVybGFwOiBpbnQgPSA2NCwNCiAgICBzcGVha2VyX3JvbGVfdG9rZW5zOiBib29sID0gRmFsc2UsDQopIC0+IGxpc3RbbGlzdFtpbnRdXToNCiAgICAiIiLthLQg6rK96rOE7JeQ7IScIOyyre2BrOulvCDrgpjriITqs6Ag66eI7KeA66eJIO2EtOydhCDri6TsnYwg7LKt7YGs7JeQIOqyueyzkCDrhKPripTri6QuDQoNCiAgICDsnbjsoJHtlZwg65GQIO2EtOydtCDrs7jrrLgg7ZWc64+EIOyViOyXkCDrk6TslrTqsIDrqbQg67CY65Oc7IucIO2VmOuCmCDsnbTsg4HsnZgg7LKt7YGs7JeQ7IScDQogICAg7ZWo6ruYIOuCmO2DgOuCnOuLpC4g7ZWc64+E66W8IOuEmOuKlCDthLTsnYAg7Yag7YGwIG92ZXJsYXDsnLzroZwg67aE7ZWg7ZWc64ukLiDshJzroZwg64uk66W4DQogICAg7YS07J2EIO2VqOq7mCDri7TsnYQg7IiYIOyXhuycvOuptCDslZ4g7YS07J2YIOuBneqzvCDrkqQg7YS07J2YIOyLnOyekeydhCDri7TsnYAg6rK96rOEIOyyre2BrOulvA0KICAgIOy2lOqwgO2VnOuLpC4g66qo65OgIOybkOusuCDthqDtgbDsnYQg67O07KG07ZWY66mwIOyyre2BrCDquLjsnbTripQg67O466y4IO2VnOuPhOulvCDrhJjsp4Ag7JWK64qU64ukLg0KICAgIGNodW5rX292ZXJsYXDsnYAg6ri0IO2EtCDrgrTrtoAg67aE7ZWg7JeQ66eMIOyCrOyaqe2VnOuLpC4g7KCV7IOBIO2EtOydgCDtlZwg7YS07J2EIOqyuey5nOuLpC4NCiAgICAiIiINCiAgICBjYXBhY2l0eSA9IG1heF9sZW5ndGggLSB0b2tlbml6ZXIubnVtX3NwZWNpYWxfdG9rZW5zX3RvX2FkZChwYWlyPUZhbHNlKQ0KICAgIGlmIGNhcGFjaXR5IDw9IDA6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1heF9sZW5ndGjripQg7Yq57IiYIO2GoO2BsCDsiJjrs7Tri6Qg7Luk7JW8IO2VqeuLiOuLpC4iKQ0KICAgIGlmIG5vdCAwIDw9IGNodW5rX292ZXJsYXAgPCBjYXBhY2l0eToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2h1bmtfb3ZlcmxhcOydgCAwIOydtOyDgeydtOupsCDrs7jrrLgg7Yag7YGwIO2VnOuPhOuztOuLpCDsnpHslYTslbwg7ZWp64uI64ukLiIpDQoNCiAgICBwaWVjZXMgPSBbXQ0KICAgIGZvciB0dXJuX2luZGV4LCB0ZXh0IGluIGVudW1lcmF0ZSgNCiAgICAgICAgYnVpbGRfZGlhbG9ndWVfdHVybnModXR0ZXJhbmNlcywgc3BlYWtlcl9yb2xlX3Rva2Vucz1zcGVha2VyX3JvbGVfdG9rZW5zKQ0KICAgICk6DQogICAgICAgIHRva2VucyA9IHRva2VuaXplcigNCiAgICAgICAgICAgIHRleHQsIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSwgdHJ1bmNhdGlvbj1GYWxzZSwNCiAgICAgICAgICAgIHJldHVybl9hdHRlbnRpb25fbWFzaz1GYWxzZSwgdmVyYm9zZT1GYWxzZSwNCiAgICAgICAgKVsiaW5wdXRfaWRzIl0NCiAgICAgICAgc3RhcnQgPSAwDQogICAgICAgIHdoaWxlIHN0YXJ0IDwgbGVuKHRva2Vucyk6DQogICAgICAgICAgICBwaWVjZXMuYXBwZW5kKCh0dXJuX2luZGV4LCB0b2tlbnNbc3RhcnQgOiBzdGFydCArIGNhcGFjaXR5XSkpDQogICAgICAgICAgICBpZiBzdGFydCArIGNhcGFjaXR5ID49IGxlbih0b2tlbnMpOg0KICAgICAgICAgICAgICAgIGJyZWFrDQogICAgICAgICAgICBzdGFydCArPSBjYXBhY2l0eSAtIGNodW5rX292ZXJsYXANCiAgICBpZiBub3QgcGllY2VzOg0KICAgICAgICByZXR1cm4gW1tdXQ0KDQogICAgY2h1bmtzID0gW10NCiAgICBjdXJyZW50ID0gW10NCiAgICBwcmV2aW91c190b2tlbnMgPSBbXQ0KICAgIHByZXZpb3VzX3R1cm4gPSBOb25lDQogICAgZm9yIHR1cm5faW5kZXgsIHRva2VucyBpbiBwaWVjZXM6DQogICAgICAgIGlmIGxlbihjdXJyZW50KSArIGxlbih0b2tlbnMpIDw9IGNhcGFjaXR5Og0KICAgICAgICAgICAgY3VycmVudCA9IGN1cnJlbnQgKyB0b2tlbnMNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGNodW5rcy5hcHBlbmQoY3VycmVudCkNCiAgICAgICAgICAgIGlmIGxlbihwcmV2aW91c190b2tlbnMpICsgbGVuKHRva2VucykgPD0gY2FwYWNpdHk6DQogICAgICAgICAgICAgICAgY3VycmVudCA9IHByZXZpb3VzX3Rva2VucyArIHRva2Vucw0KICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICAjIOuRkCDthLQg7KCE7LK06rCAIOyViCDrk6TslrTqsIDrj4Qg7KeI66y47J2YIOuBneqzvCDri7Xrs4DsnZgg7Iuc7J6R7J2EIOuztOyhtC4NCiAgICAgICAgICAgICAgICAjIOqwmeydgCDquLQg7YS0IOuCtOu2gOyXkOyEnOuKlCDsnbTrr7ggdG9rZW4gb3ZlcmxhcOydhCDsoIHsmqntlojri6QuDQogICAgICAgICAgICAgICAgaWYgcHJldmlvdXNfdHVybiAhPSB0dXJuX2luZGV4IGFuZCBjYXBhY2l0eSA+PSAyOg0KICAgICAgICAgICAgICAgICAgICBsZWZ0ID0gbWluKGxlbihwcmV2aW91c190b2tlbnMpLCBjYXBhY2l0eSAvLyAyKQ0KICAgICAgICAgICAgICAgICAgICByaWdodCA9IG1pbihsZW4odG9rZW5zKSwgY2FwYWNpdHkgLSBsZWZ0KQ0KICAgICAgICAgICAgICAgICAgICBsZWZ0ID0gbWluKGxlbihwcmV2aW91c190b2tlbnMpLCBjYXBhY2l0eSAtIHJpZ2h0KQ0KICAgICAgICAgICAgICAgICAgICBjaHVua3MuYXBwZW5kKHByZXZpb3VzX3Rva2Vuc1stbGVmdDpdICsgdG9rZW5zWzpyaWdodF0pDQogICAgICAgICAgICAgICAgY3VycmVudCA9IHRva2Vucy5jb3B5KCkNCiAgICAgICAgcHJldmlvdXNfdG9rZW5zID0gdG9rZW5zDQogICAgICAgIHByZXZpb3VzX3R1cm4gPSB0dXJuX2luZGV4DQogICAgY2h1bmtzLmFwcGVuZChjdXJyZW50KQ0KICAgIHJldHVybiBjaHVua3MNCg==","evaluate.py":"IiIi7ZWZ7Iq165CcIOyyre2BrCDrqqjrjbjsnYQg652867Ko7J20IOyeiOuKlCDrjbDsnbTthLDsl5DshJwg7Y+J6rCA7ZWc64ukLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRxZG0gaW1wb3J0IHRxZG0KCmZyb20gY29uZmlnIGltcG9ydCBEQVRBX1JPT1QsIERFRkFVTFRfQ0hFQ0tQT0lOVF9QQVRILCBPVVRQVVRfRElSLCBUQVJHRVRfU1lNUFRPTVMKZnJvbSBkYXRhX3V0aWxzIGltcG9ydCBsaXN0X2xhYmVsX2ZpbGVzCmZyb20gaW5mZXJlbmNlIGltcG9ydCBsb2FkX21vZGVsCmZyb20gbWV0cmljc191dGlscyBpbXBvcnQgY29tcHV0ZV9tdWx0aWxhYmVsX21ldHJpY3MsIHByaW50X21ldHJpY3MKZnJvbSB0cmFpbiBpbXBvcnQgTWlzc2lvbjNEYXRhc2V0LCBtYWtlX2xvYWRlcgoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGNvbGxlY3RfcHJlZGljdGlvbnMoCiAgICBtb2RlbCwKICAgIGxvYWRlciwKICAgIGRldmljZTogdG9yY2guZGV2aWNlLAogICAgdGhyZXNob2xkLAopIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgIiIiRGF0YUxvYWRlciDsoITssrTsnZgg7Ya17ZmUIOuLqOychCDsoJXri7Xqs7wg7JiI7Lih7J2EIOyImOynkS4iIiIKICAgIGZyb20gbW9kZWxfbWlzc2lvbjMgaW1wb3J0IHBvb2xfY2h1bmtfbG9naXRzCgogICAgbW9kZWwuZXZhbCgpCiAgICBhbGxfbGFiZWxzID0gW10KICAgIGFsbF9wcmVkaWN0aW9ucyA9IFtdCiAgICBmb3IgYmF0Y2ggaW4gdHFkbShsb2FkZXIsIGRlc2M9ImV2YWx1YXRlIik6CiAgICAgICAgbGFiZWxzID0gYmF0Y2hbImxhYmVscyJdLm51bXB5KCkuYXN0eXBlKG5wLmludDgpCiAgICAgICAgY2h1bmtfbG9naXRzID0gbW9kZWwoCiAgICAgICAgICAgIGJhdGNoWyJpbnB1dF9pZHMiXS50byhkZXZpY2UpLAogICAgICAgICAgICBiYXRjaFsiYXR0ZW50aW9uX21hc2siXS50byhkZXZpY2UpLAogICAgICAgICkKICAgICAgICBsb2dpdHMgPSBwb29sX2NodW5rX2xvZ2l0cygKICAgICAgICAgICAgY2h1bmtfbG9naXRzLAogICAgICAgICAgICBiYXRjaFsiY29udmVyc2F0aW9uX2lkcyJdLnRvKGRldmljZSksCiAgICAgICAgICAgIGxlbihsYWJlbHMpLAogICAgICAgICkKICAgICAgICB0aHJlc2hvbGRfdGVuc29yID0gdG9yY2guYXNfdGVuc29yKAogICAgICAgICAgICB0aHJlc2hvbGQsCiAgICAgICAgICAgIGR0eXBlPWxvZ2l0cy5kdHlwZSwKICAgICAgICAgICAgZGV2aWNlPWxvZ2l0cy5kZXZpY2UsCiAgICAgICAgKQogICAgICAgIHByZWRpY3Rpb25zID0gKAogICAgICAgICAgICB0b3JjaC5zaWdtb2lkKGxvZ2l0cykuZ3QodGhyZXNob2xkX3RlbnNvcikudG8odG9yY2guaW50OCkuY3B1KCkubnVtcHkoKQogICAgICAgICkKICAgICAgICBhbGxfbGFiZWxzLmFwcGVuZChsYWJlbHMpCiAgICAgICAgYWxsX3ByZWRpY3Rpb25zLmFwcGVuZChwcmVkaWN0aW9ucykKCiAgICBpZiBub3QgYWxsX2xhYmVsczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLtj4nqsIDtlaAgSlNPTiDtjIzsnbzsnbQg7JeG7Iq164uI64ukLiIpCiAgICByZXR1cm4gbnAudnN0YWNrKGFsbF9sYWJlbHMpLCBucC52c3RhY2soYWxsX3ByZWRpY3Rpb25zKQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJEQ0MgTWlzc2lvbiAzIOyytO2BrO2PrOyduO2KuCDtj4nqsIAiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1sYWJlbF9kaXIiLAogICAgICAgIHR5cGU9UGF0aCwKICAgICAgICBkZWZhdWx0PU5vbmUsCiAgICAgICAgaGVscD0i7KCV64u1IHN5bXB0b23snbQg65Ok7Ja0IOyeiOuKlCBKU09OIO2PtOuNlCjquLDrs7g6IOqzteyLnSBWYWxpZGF0aW9uKSIsCiAgICApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGFfcm9vdCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1EQVRBX1JPT1QpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNrcHRfcGF0aCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1ERUZBVUxUX0NIRUNLUE9JTlRfUEFUSCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tbWV0cmljc19vdXRwdXQiLAogICAgICAgIHR5cGU9UGF0aCwKICAgICAgICBkZWZhdWx0PU9VVFBVVF9ESVIgLyAidmFsaWRhdGlvbl9tZXRyaWNzLmpzb24iLAogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaF9zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9OCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdGhyZXNob2xkIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLXVzZV9jaGVja3BvaW50X3RocmVzaG9sZHMiLAogICAgICAgIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgaGVscD0i7LK07YGs7Y+s7J247Yq47JeQIOyggOyepeuQnCDrgrTrtoAg6rKA7KadIOymneyDgeuzhCDsnoTqs4TqsJIg7IKs7JqpIiwKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbWF4X3NhbXBsZXMiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBpZiBub3QgMC4wIDw9IGFyZ3MudGhyZXNob2xkIDw9IDEuMDoKICAgICAgICBwYXJzZXIuZXJyb3IoIi0tdGhyZXNob2xk64qUIDDqs7wgMSDsgqzsnbTsl6zslbwg7ZWp64uI64ukLiIpCiAgICBpZiBub3QgYXJncy5ja3B0X3BhdGguaXNfZmlsZSgpOgogICAgICAgIHBhcnNlci5lcnJvcigKICAgICAgICAgICAgZiLssrTtgaztj6zsnbjtirjqsIAg7JeG7Iq164uI64ukOiB7YXJncy5ja3B0X3BhdGh9LiDrqLzsoIAgdHJhaW4ucHnrpbwg7Iuk7ZaJ7ZWY7IS47JqULiIKICAgICAgICApCgogICAgaWYgYXJncy5sYWJlbF9kaXIgaXMgTm9uZToKICAgICAgICBmcm9tIGNvbmZpZyBpbXBvcnQgZ2V0X3ZhbF9sYWJlbF9kaXIKCiAgICAgICAgbGFiZWxfZGlyID0gZ2V0X3ZhbF9sYWJlbF9kaXIoYXJncy5kYXRhX3Jvb3QpCiAgICBlbHNlOgogICAgICAgIGxhYmVsX2RpciA9IGFyZ3MubGFiZWxfZGlyCgogICAgbGFiZWxfZmlsZXMgPSBsaXN0X2xhYmVsX2ZpbGVzKGxhYmVsX2RpcikKICAgIGlmIGFyZ3MubWF4X3NhbXBsZXMgaXMgbm90IE5vbmU6CiAgICAgICAgbGFiZWxfZmlsZXMgPSBsYWJlbF9maWxlc1s6IGFyZ3MubWF4X3NhbXBsZXNdCgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBwcmludChmIkRldmljZToge2RldmljZX0iKQogICAgcHJpbnQoZiJDaGVja3BvaW50OiB7YXJncy5ja3B0X3BhdGh9IikKICAgIHByaW50KGYiRXZhbHVhdGlvbiBkYXRhOiB7bGFiZWxfZGlyfSAoe2xlbihsYWJlbF9maWxlcyk6LH3qsbQpIikKCiAgICBtb2RlbCwgdG9rZW5pemVyLCBjaGVja3BvaW50ID0gbG9hZF9tb2RlbChhcmdzLmNrcHRfcGF0aCwgZGV2aWNlKQogICAgbWF4X2xlbmd0aCA9IGludChjaGVja3BvaW50LmdldCgibWF4X2xlbmd0aCIsIDI1NikpCiAgICBjaHVua19vdmVybGFwID0gaW50KGNoZWNrcG9pbnQuZ2V0KCJjaHVua19vdmVybGFwIiwgNjQpKQogICAgZGF0YXNldCA9IE1pc3Npb24zRGF0YXNldCgKICAgICAgICBsYWJlbF9maWxlcywKICAgICAgICB0b2tlbml6ZXIsCiAgICAgICAgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLAogICAgICAgIGNodW5rX292ZXJsYXA9Y2h1bmtfb3ZlcmxhcCwKICAgICAgICBzcGVha2VyX3JvbGVfdG9rZW5zPWJvb2woY2hlY2twb2ludC5nZXQoInNwZWFrZXJfcm9sZV90b2tlbnMiLCBGYWxzZSkpLAogICAgKQogICAgbG9hZGVyID0gbWFrZV9sb2FkZXIoZGF0YXNldCwgYXJncy5iYXRjaF9zaXplLCBGYWxzZSkKICAgIGV2YWx1YXRpb25fdGhyZXNob2xkID0gKAogICAgICAgIGNoZWNrcG9pbnQuZ2V0KCJkZWNpc2lvbl90aHJlc2hvbGRzIiwgWzAuNV0gKiBsZW4oVEFSR0VUX1NZTVBUT01TKSkKICAgICAgICBpZiBhcmdzLnVzZV9jaGVja3BvaW50X3RocmVzaG9sZHMKICAgICAgICBlbHNlIGFyZ3MudGhyZXNob2xkCiAgICApCiAgICBsYWJlbHMsIHByZWRpY3Rpb25zID0gY29sbGVjdF9wcmVkaWN0aW9ucygKICAgICAgICBtb2RlbCwKICAgICAgICBsb2FkZXIsCiAgICAgICAgZGV2aWNlLAogICAgICAgIGV2YWx1YXRpb25fdGhyZXNob2xkLAogICAgKQogICAgbWV0cmljcyA9IGNvbXB1dGVfbXVsdGlsYWJlbF9tZXRyaWNzKGxhYmVscywgcHJlZGljdGlvbnMsIFRBUkdFVF9TWU1QVE9NUykKICAgIG1ldHJpY3MudXBkYXRlKAogICAgICAgIHsKICAgICAgICAgICAgInRocmVzaG9sZCI6IGV2YWx1YXRpb25fdGhyZXNob2xkLAogICAgICAgICAgICAidGhyZXNob2xkX3NvdXJjZSI6ICgKICAgICAgICAgICAgICAgICJjaGVja3BvaW50X2ludGVybmFsX3ZhbGlkYXRpb24iCiAgICAgICAgICAgICAgICBpZiBhcmdzLnVzZV9jaGVja3BvaW50X3RocmVzaG9sZHMKICAgICAgICAgICAgICAgIGVsc2UgImNvbW1hbmRfbGluZV9maXhlZCIKICAgICAgICAgICAgKSwKICAgICAgICAgICAgInNwZWFrZXJfcm9sZV90b2tlbnMiOiBib29sKGNoZWNrcG9pbnQuZ2V0KCJzcGVha2VyX3JvbGVfdG9rZW5zIiwgRmFsc2UpKSwKICAgICAgICAgICAgImNoZWNrcG9pbnQiOiBzdHIoYXJncy5ja3B0X3BhdGgucmVzb2x2ZSgpKSwKICAgICAgICAgICAgImxhYmVsX2RpciI6IHN0cihsYWJlbF9kaXIucmVzb2x2ZSgpKSwKICAgICAgICAgICAgIm1heF9sZW5ndGgiOiBtYXhfbGVuZ3RoLAogICAgICAgICAgICAiY2h1bmtfb3ZlcmxhcCI6IGNodW5rX292ZXJsYXAsCiAgICAgICAgICAgICJjaHVua19zdHJhdGVneSI6IGNoZWNrcG9pbnRbImNodW5rX3N0cmF0ZWd5Il0sCiAgICAgICAgfQogICAgKQogICAgcHJpbnRfbWV0cmljcyhtZXRyaWNzKQoKICAgIGFyZ3MubWV0cmljc19vdXRwdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGFyZ3MubWV0cmljc19vdXRwdXQud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKG1ldHJpY3MsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLAogICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICApCiAgICBwcmludChmIlxu7KeA7ZGcIOyggOyepToge2FyZ3MubWV0cmljc19vdXRwdXR9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==","experiment_utils.py":"IiIi7J6s7ZiEIOqwgOuKpe2VnCBtdWx0aWxhYmVsIHNwbGl06rO8IOyghOyytCDrhbjstpwg67O07J6lIHNhbXBsZXIuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aApmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgSXRlcmF0b3IsIFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgU2FtcGxlcgoKZnJvbSBzcGxpdF91dGlscyBpbXBvcnQgaXRlcmF0aXZlX211bHRpbGFiZWxfc3BsaXQKCgpjbGFzcyBHdWFyYW50ZWVkRnVsbFBhc3NTYW1wbGVyKFNhbXBsZXJbaW50XSk6CiAgICAiIiLrqqjrk6Ag7IOY7ZSM7J2EIDHtmowg7Y+s7ZWo7ZWY6rOgIOqwgOykkey5mOydmCDstIjqs7zrtoTrp4wg7LaU6rCAIOy2lOy2nO2VnOuLpC4KCiAgICDsmIjrpbwg65Ok7Ja0IGhhcmQtbmVnYXRpdmUg6rCA7KSR7LmY6rCAIDEuNeydtOuptCDquLDrs7ggMe2ajCDrhbjstpzsnYAg67O07J6l65CY6rOgLAogICAgaGFyZC1uZWdhdGl2ZSDsp5Htlakg7YGs6riw7J2YIOyVvSA1MCXrp4ztgbwg7LaU6rCAIOyDmO2UjOydtCBlcG9jaOyXkCDshJ7snbjri6QuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgc2FtcGxlX3dlaWdodHM6IFNlcXVlbmNlW2Zsb2F0XSwgc2VlZDogaW50ID0gNDIpOgogICAgICAgIHdlaWdodHMgPSB0b3JjaC5hc190ZW5zb3Ioc2FtcGxlX3dlaWdodHMsIGR0eXBlPXRvcmNoLmRvdWJsZSkKICAgICAgICBpZiB3ZWlnaHRzLm5kaW0gIT0gMSBvciBsZW4od2VpZ2h0cykgPT0gMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2FtcGxlX3dlaWdodHPripQg67mE7Ja0IOyeiOyngCDslYrsnYAgMeywqOybkOydtOyWtOyVvCDtlanri4jri6QuIikKICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUod2VpZ2h0cykuYWxsKCkgb3IgdG9yY2guYW55KHdlaWdodHMgPCAxLjApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLsoITssrQg64W47LacIOuztOyepSBzYW1wbGVy7J2YIOqwgOykkey5mOuKlCDsnKDtlZztlZwgMSDsnbTsg4HsnbTslrTslbwg7ZWp64uI64ukLiIpCiAgICAgICAgc2VsZi53ZWlnaHRzID0gd2VpZ2h0cwogICAgICAgIHNlbGYuc2VlZCA9IGludChzZWVkKQogICAgICAgIHNlbGYuZXBvY2ggPSAwCiAgICAgICAgc2VsZi5leHRyYV9jb3VudCA9IGludChtYXRoLmZsb29yKGZsb2F0KHRvcmNoLnN1bSh3ZWlnaHRzIC0gMS4wKS5pdGVtKCkpICsgMC41KSkKCiAgICBkZWYgX19sZW5fXyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLndlaWdodHMpICsgc2VsZi5leHRyYV9jb3VudAoKICAgIGRlZiBfX2l0ZXJfXyhzZWxmKSAtPiBJdGVyYXRvcltpbnRdOgogICAgICAgIGdlbmVyYXRvciA9IHRvcmNoLkdlbmVyYXRvcigpCiAgICAgICAgZ2VuZXJhdG9yLm1hbnVhbF9zZWVkKHNlbGYuc2VlZCArIHNlbGYuZXBvY2gpCiAgICAgICAgc2VsZi5lcG9jaCArPSAxCgogICAgICAgIGJhc2UgPSB0b3JjaC5yYW5kcGVybShsZW4oc2VsZi53ZWlnaHRzKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikudG9saXN0KCkKICAgICAgICBpZiBzZWxmLmV4dHJhX2NvdW50OgogICAgICAgICAgICBleHRyYV93ZWlnaHRzID0gc2VsZi53ZWlnaHRzIC0gMS4wCiAgICAgICAgICAgIHBvc2l0aXZlID0gaW50KHRvcmNoLmNvdW50X25vbnplcm8oZXh0cmFfd2VpZ2h0cykuaXRlbSgpKQogICAgICAgICAgICBpZiBwb3NpdGl2ZSA9PSAwOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCLstpTqsIAg7LaU7LacIOyImOuKlCDslpHsiJjsnbjrjbAg7LSI6rO8IOqwgOykkey5mOqwgCDsl4bsirXri4jri6QuIikKICAgICAgICAgICAgZXh0cmFzID0gdG9yY2gubXVsdGlub21pYWwoCiAgICAgICAgICAgICAgICBleHRyYV93ZWlnaHRzLAogICAgICAgICAgICAgICAgc2VsZi5leHRyYV9jb3VudCwKICAgICAgICAgICAgICAgIHJlcGxhY2VtZW50PXNlbGYuZXh0cmFfY291bnQgPiBwb3NpdGl2ZSwKICAgICAgICAgICAgICAgIGdlbmVyYXRvcj1nZW5lcmF0b3IsCiAgICAgICAgICAgICkudG9saXN0KCkKICAgICAgICAgICAgYmFzZS5leHRlbmQoZXh0cmFzKQogICAgICAgICAgICBvcmRlciA9IHRvcmNoLnJhbmRwZXJtKGxlbihiYXNlKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikudG9saXN0KCkKICAgICAgICAgICAgYmFzZSA9IFtiYXNlW2luZGV4XSBmb3IgaW5kZXggaW4gb3JkZXJdCiAgICAgICAgcmV0dXJuIGl0ZXIoYmFzZSkKCgpkZWYgc2FtcGxlcl9jb25maWcoc2FtcGxlX3dlaWdodHM6IFNlcXVlbmNlW2Zsb2F0XSkgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICAiIiLssrTtgaztj6zsnbjtirjsl5Ag6riw66Gd7ZWgIOyghOyytCDrhbjstpwg67O07J6lIHNhbXBsZXIg7JqU7JW9LiIiIgogICAgd2VpZ2h0cyA9IG5wLmFzYXJyYXkoc2FtcGxlX3dlaWdodHMsIGR0eXBlPWZsb2F0KQogICAgZXh0cmFfc2FtcGxlcyA9IGludChtYXRoLmZsb29yKGZsb2F0KG5wLnN1bSh3ZWlnaHRzIC0gMS4wKSkgKyAwLjUpKQogICAgcmV0dXJuIHsKICAgICAgICAibmFtZSI6ICJndWFyYW50ZWVkX2Z1bGxfcGFzc19oYXJkX25lZ2F0aXZlX3NhbXBsZXIiLAogICAgICAgICJndWFyYW50ZWVzX2V2ZXJ5X3NhbXBsZV9lYWNoX2Vwb2NoIjogVHJ1ZSwKICAgICAgICAiYmFzZV9zYW1wbGVzIjogaW50KGxlbih3ZWlnaHRzKSksCiAgICAgICAgImV4dHJhX3NhbXBsZXMiOiBleHRyYV9zYW1wbGVzLAogICAgICAgICJlcG9jaF9udW1fc2FtcGxlcyI6IGludChsZW4od2VpZ2h0cykgKyBleHRyYV9zYW1wbGVzKSwKICAgICAgICAibWF4X3dlaWdodCI6IGZsb2F0KHdlaWdodHMubWF4KCkpIGlmIGxlbih3ZWlnaHRzKSBlbHNlIDEuMCwKICAgICAgICAicmVwbGFjZW1lbnQiOiAiZXh0cmFzX29ubHlfd2hlbl9yZXF1aXJlZCIsCiAgICB9Cgo=","hard_negative_utils.py":"IiIiSGFyZC1uZWdhdGl2ZSBtYW5pZmVzdCDqsoDspp3qs7wg7J6s7ZiEIOqwgOuKpe2VnCDsg5jtlIwg6rCA7KSR7LmYIOyDneyEsS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IG1hdGgKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgbG9hZF9oYXJkX25lZ2F0aXZlX3NhbXBsaW5nKAogICAgbWFuaWZlc3RfcGF0aDogUGF0aCwKICAgIHRyYWluX2ZpbGVzOiBsaXN0W1BhdGhdLAogICAgdGFyZ2V0X3N5bXB0b21zOiBsaXN0W3N0cl0gfCB0dXBsZVtzdHIsIC4uLl0sCiAgICBmb2N1c19zeW1wdG9tczogbGlzdFtzdHJdIHwgdHVwbGVbc3RyLCAuLi5dLAogICAgb3ZlcnNhbXBsZV9mYWN0b3I6IGZsb2F0LAopIC0+IHR1cGxlW2xpc3RbZmxvYXRdLCBkaWN0XToKICAgICIiIuqygO2GoOuQnCBGUCBtYW5pZmVzdOulvCDtlZnsirUg7Ya17ZmU67OEIFdlaWdodGVkUmFuZG9tU2FtcGxlciDqsIDspJHsuZjroZwg67OA7ZmYLgoKICAgIG1hbmlmZXN07JeQ64qUIOy1nOyGjCBgYGZpbGVfbmFtZWBg6rO8IGBgc3ltcHRvbWBgIOyXtOydtCDtlYTsmpTtlZjri6QuIGBgZXJyb3JfdHlwZWBgCiAgICDsl7TsnbQg7J6I7Jy866m0IEZQ66eMIOyCrOyaqe2VmOqzoCwgYGBldmlkZW5jZV9idWNrZXRgYCDsl7TsnbQg7J6I7Jy866m0CiAgICBgYOuLqOyEnCDrtoDsobEg65iQ64qUIOuqqO2YuGBg7J24IO2WieunjCDsgqzsmqntlZzri6QuIOuCtOu2gCDqsoDspp0g7YyM7J286rO8IG1hbmlmZXN07J2YIOuLpOuluAogICAgc3BsaXQg7YyM7J287J2AIGBgdHJhaW5fZmlsZXNgYOyXkCDsl4bsnLzrr4DroZwg7J6Q64+Z7Jy866GcIOygnOyZuOuQnOuLpC4KICAgICIiIgogICAgcGF0aCA9IFBhdGgobWFuaWZlc3RfcGF0aCkKICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiaGFyZC1uZWdhdGl2ZSBtYW5pZmVzdOqwgCDsl4bsirXri4jri6Q6IHtwYXRofSIpCiAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShvdmVyc2FtcGxlX2ZhY3Rvcikgb3Igb3ZlcnNhbXBsZV9mYWN0b3IgPCAxLjA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaGFyZC1uZWdhdGl2ZSBvdmVyc2FtcGxlIGZhY3RvcuuKlCDsnKDtlZztlZwgMSDsnbTsg4HsnbTslrTslbwg7ZWp64uI64ukLiIpCiAgICBpZiBub3QgdHJhaW5fZmlsZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaGFyZC1uZWdhdGl2ZSDqsIDspJHsuZjrpbwg66eM65OkIO2VmeyKtSDtjIzsnbzsnbQg7JeG7Iq164uI64ukLiIpCgogICAgdGFyZ2V0X3NldCA9IHNldCh0YXJnZXRfc3ltcHRvbXMpCiAgICBmb2N1cyA9IGxpc3QoZGljdC5mcm9ta2V5cyhmb2N1c19zeW1wdG9tcykpCiAgICBpZiBub3QgZm9jdXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaGFyZC1uZWdhdGl2ZSBmb2N1cyBzeW1wdG9t7J2AIDHqsJwg7J207IOB7J207Ja07JW8IO2VqeuLiOuLpC4iKQogICAgdW5rbm93biA9IHNvcnRlZChzZXQoZm9jdXMpIC0gdGFyZ2V0X3NldCkKICAgIGlmIHVua25vd246CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIuyVjCDsiJgg7JeG64qUIGhhcmQtbmVnYXRpdmUg7Kad7IOBOiB7dW5rbm93bn0iKQogICAgZm9jdXNfc2V0ID0gc2V0KGZvY3VzKQoKICAgIHdpdGggcGF0aC5vcGVuKCJyIiwgZW5jb2Rpbmc9InV0Zi04LXNpZyIsIG5ld2xpbmU9IiIpIGFzIGZpbGU6CiAgICAgICAgcmVhZGVyID0gY3N2LkRpY3RSZWFkZXIoZmlsZSkKICAgICAgICBmaWVsZG5hbWVzID0gc2V0KHJlYWRlci5maWVsZG5hbWVzIG9yIFtdKQogICAgICAgIG1pc3NpbmcgPSBzb3J0ZWQoeyJmaWxlX25hbWUiLCAic3ltcHRvbSJ9IC0gZmllbGRuYW1lcykKICAgICAgICBpZiBtaXNzaW5nOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiaGFyZC1uZWdhdGl2ZSBtYW5pZmVzdCDtlYTsiJgg7Je07J20IOyXhuyKteuLiOuLpDoge21pc3Npbmd9IikKICAgICAgICByYXdfcm93cyA9IGxpc3QocmVhZGVyKQoKICAgIGFjY2VwdGVkX3BhaXJzOiBzZXRbdHVwbGVbc3RyLCBzdHJdXSA9IHNldCgpCiAgICByZWplY3RlZF9ub25fZnAgPSAwCiAgICByZWplY3RlZF9jbGVhcl9ldmlkZW5jZSA9IDAKICAgIHJlamVjdGVkX290aGVyX3N5bXB0b20gPSAwCiAgICBmb3Igcm93IGluIHJhd19yb3dzOgogICAgICAgIGZpbGVfbmFtZSA9IHN0cihyb3cuZ2V0KCJmaWxlX25hbWUiLCAiIikpLnN0cmlwKCkKICAgICAgICBzeW1wdG9tID0gc3RyKHJvdy5nZXQoInN5bXB0b20iLCAiIikpLnN0cmlwKCkKICAgICAgICBlcnJvcl90eXBlID0gc3RyKHJvdy5nZXQoImVycm9yX3R5cGUiLCAiRlAiKSkuc3RyaXAoKS51cHBlcigpCiAgICAgICAgZXZpZGVuY2VfYnVja2V0ID0gc3RyKHJvdy5nZXQoImV2aWRlbmNlX2J1Y2tldCIsICIiKSkuc3RyaXAoKQogICAgICAgIGlmIGVycm9yX3R5cGUgIT0gIkZQIjoKICAgICAgICAgICAgcmVqZWN0ZWRfbm9uX2ZwICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBldmlkZW5jZV9idWNrZXQgYW5kIGV2aWRlbmNlX2J1Y2tldCAhPSAi64uo7IScIOu2gOyhsSDrmJDripQg66qo7Zi4IjoKICAgICAgICAgICAgcmVqZWN0ZWRfY2xlYXJfZXZpZGVuY2UgKz0gMQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIHN5bXB0b20gbm90IGluIGZvY3VzX3NldDoKICAgICAgICAgICAgcmVqZWN0ZWRfb3RoZXJfc3ltcHRvbSArPSAxCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgZmlsZV9uYW1lOgogICAgICAgICAgICBhY2NlcHRlZF9wYWlycy5hZGQoKGZpbGVfbmFtZSwgc3ltcHRvbSkpCgogICAgdHJhaW5fbmFtZXMgPSBbUGF0aChmaWxlKS5uYW1lIGZvciBmaWxlIGluIHRyYWluX2ZpbGVzXQogICAgdHJhaW5fbmFtZV9zZXQgPSBzZXQodHJhaW5fbmFtZXMpCiAgICBtYXRjaGVkX3BhaXJzID0gc29ydGVkKHBhaXIgZm9yIHBhaXIgaW4gYWNjZXB0ZWRfcGFpcnMgaWYgcGFpclswXSBpbiB0cmFpbl9uYW1lX3NldCkKICAgIG1hdGNoZWRfbmFtZXMgPSB7ZmlsZV9uYW1lIGZvciBmaWxlX25hbWUsIF8gaW4gbWF0Y2hlZF9wYWlyc30KICAgIGlmIG5vdCBtYXRjaGVkX25hbWVzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICJtYW5pZmVzdOyXkOyEnCDtmITsnqwg64K067aAIO2VmeyKtSBzcGxpdOqzvCDsnbzsuZjtlZjripQgaGFyZC1uZWdhdGl2ZSDtjIzsnbzsnYQg7LC+7KeAIOuqu+2WiOyKteuLiOuLpC4gIgogICAgICAgICAgICAi64+Z7J287ZWcIHNhbXBsZV9yYXRpbywgc2VlZOyZgCBUcmFpbmluZyBzcGxpdOycvOuhnCDri6Tsi5wg7LGE6rW07ZWY7IS47JqULiIKICAgICAgICApCgogICAgd2VpZ2h0cyA9IFtvdmVyc2FtcGxlX2ZhY3RvciBpZiBuYW1lIGluIG1hdGNoZWRfbmFtZXMgZWxzZSAxLjAgZm9yIG5hbWUgaW4gdHJhaW5fbmFtZXNdCiAgICBzeW1wdG9tX2NvdW50cyA9IENvdW50ZXIoc3ltcHRvbSBmb3IgXywgc3ltcHRvbSBpbiBtYXRjaGVkX3BhaXJzKQogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYocGF0aC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpCiAgICBjb25maWcgPSB7CiAgICAgICAgIm5hbWUiOiAiaGFyZF9uZWdhdGl2ZV93ZWlnaHRlZF9zYW1wbGVyIiwKICAgICAgICAibWFuaWZlc3QiOiBzdHIocGF0aC5yZXNvbHZlKCkpLAogICAgICAgICJtYW5pZmVzdF9zaGEyNTYiOiBkaWdlc3QsCiAgICAgICAgIm92ZXJzYW1wbGVfZmFjdG9yIjogZmxvYXQob3ZlcnNhbXBsZV9mYWN0b3IpLAogICAgICAgICJmb2N1c19zeW1wdG9tcyI6IGZvY3VzLAogICAgICAgICJudW1fbWFuaWZlc3Rfcm93cyI6IGxlbihyYXdfcm93cyksCiAgICAgICAgIm51bV91bmlxdWVfY2FuZGlkYXRlX3BhaXJzIjogbGVuKGFjY2VwdGVkX3BhaXJzKSwKICAgICAgICAibnVtX21hdGNoZWRfcGFpcnMiOiBsZW4obWF0Y2hlZF9wYWlycyksCiAgICAgICAgIm51bV9tYXRjaGVkX2NvbnZlcnNhdGlvbnMiOiBsZW4obWF0Y2hlZF9uYW1lcyksCiAgICAgICAgIm1hdGNoZWRfcGFpcnNfYnlfc3ltcHRvbSI6IHsKICAgICAgICAgICAgc3ltcHRvbTogaW50KHN5bXB0b21fY291bnRzLmdldChzeW1wdG9tLCAwKSkgZm9yIHN5bXB0b20gaW4gZm9jdXMKICAgICAgICB9LAogICAgICAgICJudW1fdW5tYXRjaGVkX2NhbmRpZGF0ZV9wYWlycyI6IGxlbihhY2NlcHRlZF9wYWlycykgLSBsZW4obWF0Y2hlZF9wYWlycyksCiAgICAgICAgInJlamVjdGVkX25vbl9mcF9yb3dzIjogcmVqZWN0ZWRfbm9uX2ZwLAogICAgICAgICJyZWplY3RlZF9jbGVhcl9ldmlkZW5jZV9yb3dzIjogcmVqZWN0ZWRfY2xlYXJfZXZpZGVuY2UsCiAgICAgICAgInJlamVjdGVkX290aGVyX3N5bXB0b21fcm93cyI6IHJlamVjdGVkX290aGVyX3N5bXB0b20sCiAgICAgICAgImVwb2NoX251bV9zYW1wbGVzIjogbGVuKHRyYWluX2ZpbGVzKSwKICAgICAgICAicmVwbGFjZW1lbnQiOiBUcnVlLAogICAgfQogICAgcmV0dXJuIHdlaWdodHMsIGNvbmZpZwo=","inference.py":"IiIiRENDIE1pc3Npb24gMyDsoITsmqkg7LaU66GgIOuwjyBDU1Yg7IOd7ISxIOyKpO2BrOumve2KuC4KCuyCrOyaqSDsmIg6CiAgcHl0aG9uIGluZmVyZW5jZS5weSBcCiAgICAgIC0tbGFiZWxfZGlyICJWYWxpZGF0aW9uLzIu652867Ko66eB642w7J207YSwL1ZMX+yEnOyauF/rjbDsnbTthLAiIFwKICAgICAgLS1ja3B0X3BhdGggY2hlY2twb2ludHMvbWlzc2lvbjNfY2h1bmtlZC5wdCBcCiAgICAgIC0tb3V0cHV0IG91dHB1dHMvbWlzc2lvbjNfY2h1bmtlZC5jc3YKIiIiCgppbXBvcnQgYXJncGFyc2UKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Rva2VuaXplcgoKZnJvbSBjb25maWcgaW1wb3J0ICgKICAgIENIVU5LX1NUUkFURUdZLAogICAgREFUQV9ST09ULAogICAgREVGQVVMVF9DSEVDS1BPSU5UX1BBVEgsCiAgICBERUZBVUxUX1NVQk1JU1NJT05fUEFUSCwKICAgIFRBUkdFVF9TWU1QVE9NUywKICAgIGdldF92YWxfbGFiZWxfZGlyLAopCmZyb20gZGF0YV91dGlscyBpbXBvcnQgKAogICAgbGlzdF9sYWJlbF9maWxlcywKICAgIGxvYWRfbGFiZWwsCiAgICBtdWx0aWhvdF90b19zeW1wdG9tX3N0ciwKICAgIHRva2VuaXplX2RpYWxvZ3VlX2NodW5rcywKKQpmcm9tIG1vZGVsX21pc3Npb24zIGltcG9ydCBTeW1wdG9tQ2xhc3NpZmllciwgcG9vbF9jaHVua19sb2dpdHMKCgpkZWYgbG9hZF9tb2RlbChjaGVja3BvaW50X3BhdGg6IFBhdGgsIGRldmljZTogdG9yY2guZGV2aWNlKToKICAgICIiIuuvuOyFmCAzIOyytO2BrO2PrOyduO2KuOuhnCDrqqjrjbjqs7wg7Yag7YGs64KY7J207KCAIOq1rOyEsS4iIiIKICAgIGNoZWNrcG9pbnQgPSB0b3JjaC5sb2FkKAogICAgICAgIGNoZWNrcG9pbnRfcGF0aCwKICAgICAgICBtYXBfbG9jYXRpb249ZGV2aWNlLAogICAgICAgIHdlaWdodHNfb25seT1GYWxzZSwKICAgICkKICAgIGlmIGNoZWNrcG9pbnQuZ2V0KCJjaHVua19zdHJhdGVneSIpICE9IENIVU5LX1NUUkFURUdZOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuyyre2BrCDrsKnsi53snbQg64uk66aF64uI64ukLiDsnbQg7Y+0642U7J2YIHRyYWluLnB566GcIOyDiOuhnCDtlZnsirXtlZwg7LK07YGs7Y+s7J247Yq466W8IOyCrOyaqe2VmOyEuOyalC4iKQogICAgbW9kZWxfbmFtZSA9IGNoZWNrcG9pbnQuZ2V0KCJtb2RlbF9uYW1lIiwgImJlb21pL2tjYmVydC1iYXNlIikKICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX25hbWUpCiAgICBtb2RlbCA9IFN5bXB0b21DbGFzc2lmaWVyKG1vZGVsX25hbWUsIGxlbihUQVJHRVRfU1lNUFRPTVMpKQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGNoZWNrcG9pbnRbIm1vZGVsX3N0YXRlX2RpY3QiXSkKICAgIHJldHVybiBtb2RlbC50byhkZXZpY2UpLmV2YWwoKSwgdG9rZW5pemVyLCBjaGVja3BvaW50CgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgaW5mZXJfbWlzc2lvbjMoCiAgICBtb2RlbCwKICAgIHRva2VuaXplciwKICAgIGxhYmVsX2RpcjogUGF0aCwKICAgIGRldmljZTogdG9yY2guZGV2aWNlLAogICAgbWF4X2xlbmd0aDogaW50LAogICAgY2h1bmtfb3ZlcmxhcDogaW50LAogICAgc3BlYWtlcl9yb2xlX3Rva2VuczogYm9vbCA9IEZhbHNlLAogICAgdGhyZXNob2xkcz1Ob25lLAopOgogICAgIiIi7Ya17ZmU67OEIDnsooUg7Kad7IOB7J2EIOy2lOuhoO2VmOyXrCBEYXRhRnJhbWXsnLzroZwg67CY7ZmYLiIiIgogICAgcm93cyA9IFtdCgogICAgZm9yIGxhYmVsX3BhdGggaW4gbGlzdF9sYWJlbF9maWxlcyhsYWJlbF9kaXIpOgogICAgICAgIGxhYmVsID0gbG9hZF9sYWJlbChsYWJlbF9wYXRoKQogICAgICAgIGVuY29kZWQgPSB0b2tlbml6ZV9kaWFsb2d1ZV9jaHVua3MoCiAgICAgICAgICAgIHRva2VuaXplciwKICAgICAgICAgICAgbGFiZWwuZ2V0KCJ1dHRlcmFuY2VzIiwgW10pLAogICAgICAgICAgICBtYXhfbGVuZ3RoPW1heF9sZW5ndGgsCiAgICAgICAgICAgIGNodW5rX292ZXJsYXA9Y2h1bmtfb3ZlcmxhcCwKICAgICAgICAgICAgc3BlYWtlcl9yb2xlX3Rva2Vucz1zcGVha2VyX3JvbGVfdG9rZW5zLAogICAgICAgICkKICAgICAgICBjaHVua19sb2dpdHMgPSBtb2RlbCgKICAgICAgICAgICAgZW5jb2RlZFsiaW5wdXRfaWRzIl0udG8oZGV2aWNlKSwKICAgICAgICAgICAgZW5jb2RlZFsiYXR0ZW50aW9uX21hc2siXS50byhkZXZpY2UpLAogICAgICAgICkKICAgICAgICBjb252ZXJzYXRpb25faWRzID0gdG9yY2guemVyb3MoCiAgICAgICAgICAgIGNodW5rX2xvZ2l0cy5zaGFwZVswXSwKICAgICAgICAgICAgZHR5cGU9dG9yY2gubG9uZywKICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICApCiAgICAgICAgbG9naXRzID0gcG9vbF9jaHVua19sb2dpdHMoY2h1bmtfbG9naXRzLCBjb252ZXJzYXRpb25faWRzLCAxKQogICAgICAgIHRocmVzaG9sZF90ZW5zb3IgPSB0b3JjaC5hc190ZW5zb3IoCiAgICAgICAgICAgIHRocmVzaG9sZHMgaWYgdGhyZXNob2xkcyBpcyBub3QgTm9uZSBlbHNlIDAuNSwKICAgICAgICAgICAgZHR5cGU9bG9naXRzLmR0eXBlLAogICAgICAgICAgICBkZXZpY2U9bG9naXRzLmRldmljZSwKICAgICAgICApCiAgICAgICAgcHJlZGljdGlvbnMgPSAoCiAgICAgICAgICAgICh0b3JjaC5zaWdtb2lkKGxvZ2l0cykgPiB0aHJlc2hvbGRfdGVuc29yKQogICAgICAgICAgICAuaW50KCkKICAgICAgICAgICAgLmNwdSgpCiAgICAgICAgICAgIC5udW1weSgpWzBdCiAgICAgICAgICAgIC50b2xpc3QoKQogICAgICAgICkKICAgICAgICByb3dzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImxhYmVsIGZpbGUgbmFtZSI6IGxhYmVsX3BhdGgubmFtZSwKICAgICAgICAgICAgICAgICJzeW1wdG9tIjogbXVsdGlob3RfdG9fc3ltcHRvbV9zdHIocHJlZGljdGlvbnMpLAogICAgICAgICAgICB9CiAgICAgICAgKQoKICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iRENDIE1pc3Npb24gMyDstpTroaAiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhX3Jvb3QiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9REFUQV9ST09UKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1sYWJlbF9kaXIiLAogICAgICAgIHR5cGU9UGF0aCwKICAgICAgICBkZWZhdWx0PU5vbmUsCiAgICAgICAgaGVscD0i7IOd65617ZWY66m0IGRhdGFfcm9vdCDslYTrnpgg6rO17IudIFZhbGlkYXRpb24g652867KoIO2PtOuNlCDsgqzsmqkiLAogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1ja3B0X3BhdGgiLAogICAgICAgIHR5cGU9UGF0aCwKICAgICAgICBkZWZhdWx0PURFRkFVTFRfQ0hFQ0tQT0lOVF9QQVRILAogICAgICAgIGhlbHA9IuuvuOyFmCAzIOyytO2BrO2PrOyduO2KuCDqsr3roZwiLAogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1vdXRwdXQiLAogICAgICAgIHR5cGU9UGF0aCwKICAgICAgICBkZWZhdWx0PURFRkFVTFRfU1VCTUlTU0lPTl9QQVRILAogICAgICAgIGhlbHA9IuqysOqzvCBDU1Yg7KCA7J6lIOqyveuhnCIsCiAgICApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLW1heF9sZW5ndGgiLAogICAgICAgIHR5cGU9aW50LAogICAgICAgIGRlZmF1bHQ9Tm9uZSwKICAgICAgICBoZWxwPSLssrTtgaztj6zsnbjtirgg7ISk7KCV7J2EIOuNruyWtOyTuCDssq3tgawg6ri47J20IiwKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tY2h1bmtfb3ZlcmxhcCIsCiAgICAgICAgdHlwZT1pbnQsCiAgICAgICAgZGVmYXVsdD1Ob25lLAogICAgICAgIGhlbHA9IuyytO2BrO2PrOyduO2KuCDshKTsoJXsnYQg642u7Ja07JO4IOq4tCDthLQg64K067aAIOykkeuztSDthqDtgbAg7IiYIiwKICAgICkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgbGFiZWxfZGlyID0gYXJncy5sYWJlbF9kaXIgb3IgZ2V0X3ZhbF9sYWJlbF9kaXIoYXJncy5kYXRhX3Jvb3QpCiAgICBjaGVja3BvaW50X3BhdGggPSBhcmdzLmNrcHRfcGF0aAogICAgb3V0cHV0X3BhdGggPSBhcmdzLm91dHB1dAogICAgb3V0cHV0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1vZGVsLCB0b2tlbml6ZXIsIGNoZWNrcG9pbnQgPSBsb2FkX21vZGVsKGNoZWNrcG9pbnRfcGF0aCwgZGV2aWNlKQogICAgbWF4X2xlbmd0aCA9IGFyZ3MubWF4X2xlbmd0aCBvciBjaGVja3BvaW50LmdldCgibWF4X2xlbmd0aCIsIDI1NikKICAgIGNodW5rX292ZXJsYXAgPSAoCiAgICAgICAgYXJncy5jaHVua19vdmVybGFwCiAgICAgICAgaWYgYXJncy5jaHVua19vdmVybGFwIGlzIG5vdCBOb25lCiAgICAgICAgZWxzZSBjaGVja3BvaW50LmdldCgiY2h1bmtfb3ZlcmxhcCIsIDY0KQogICAgKQogICAgaWYgbWF4X2xlbmd0aCA+IHRva2VuaXplci5tb2RlbF9tYXhfbGVuZ3RoOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYibWF4X2xlbmd0aD17bWF4X2xlbmd0aH3qsIAg66qo6424IOyngOybkCDquLjsnbQgIgogICAgICAgICAgICBmInt0b2tlbml6ZXIubW9kZWxfbWF4X2xlbmd0aH3rpbwg7LSI6rO87ZWp64uI64ukLiIKICAgICAgICApCiAgICBkYXRhZnJhbWUgPSBpbmZlcl9taXNzaW9uMygKICAgICAgICBtb2RlbCwKICAgICAgICB0b2tlbml6ZXIsCiAgICAgICAgbGFiZWxfZGlyLAogICAgICAgIGRldmljZSwKICAgICAgICBtYXhfbGVuZ3RoLAogICAgICAgIGNodW5rX292ZXJsYXAsCiAgICAgICAgc3BlYWtlcl9yb2xlX3Rva2Vucz1ib29sKGNoZWNrcG9pbnQuZ2V0KCJzcGVha2VyX3JvbGVfdG9rZW5zIiwgRmFsc2UpKSwKICAgICAgICB0aHJlc2hvbGRzPWNoZWNrcG9pbnQuZ2V0KCJkZWNpc2lvbl90aHJlc2hvbGRzIiwgMC41KSwKICAgICkKICAgIGRhdGFmcmFtZS50b19jc3Yob3V0cHV0X3BhdGgsIGluZGV4PUZhbHNlLCBlbmNvZGluZz0idXRmLTgtc2lnIikKICAgIHByaW50KGYiTWlzc2lvbiAzIOy2lOuhoCDsmYTro4w6IHtsZW4oZGF0YWZyYW1lKX3qsbQg4oaSIHtvdXRwdXRfcGF0aH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK","loss_utils.py":"IiIi7ZWZ7Iq1IO2Gte2ZlOydmCDrnbzrsqgg67mI64+E66GcIOyWkeyEsSDshpDsi6Qg6rCA7KSR7LmY66W8IOqzhOyCsO2VnOuLpC4iIiIKCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCgoKZGVmIGNvbXB1dGVfcG9zX3dlaWdodHMobGFiZWxfZmlsZXMsIHRhcmdldF9zeW1wdG9tcywgc2NhbGU9MS4wLCBtYXhfd2VpZ2h0PU5vbmUpOgogICAgIiIi64K067aAIOqygOymneydhCDsoJzsmbjtlZwg7Ya17ZmUIO2MjOydvOunjCDsoITri6ztlZzri6QuIOyyre2BrCDsiJjripQg7IS47KeAIOyViuuKlOuLpC4iIiIKICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHNjYWxlKSBvciBzY2FsZSA8IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicG9zX3dlaWdodCBzY2FsZeydgCDsnKDtlZztlZwgMCDsnbTsg4HsnZgg6rCS7J207Ja07JW8IO2VqeuLiOuLpC4iKQogICAgaWYgbWF4X3dlaWdodCBpcyBub3QgTm9uZSBhbmQgKG5vdCBtYXRoLmlzZmluaXRlKG1heF93ZWlnaHQpIG9yIG1heF93ZWlnaHQgPCAxKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwb3Nfd2VpZ2h0IOyDge2VnOydgCDsnKDtlZztlZwgMSDsnbTsg4HsnZgg6rCS7J207Ja07JW8IO2VqeuLiOuLpC4iKQogICAgaWYgbm90IGxhYmVsX2ZpbGVzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuqwgOykkey5mOulvCDqs4TsgrDtlaAg7ZWZ7Iq1IOuNsOydtO2EsOqwgCDsl4bsirXri4jri6QuIikKICAgIHBvc2l0aXZlcyA9IHRvcmNoLnplcm9zKGxlbih0YXJnZXRfc3ltcHRvbXMpLCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgZm9yIHBhdGggaW4gbGFiZWxfZmlsZXM6CiAgICAgICAgbGFiZWwgPSBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIHN5bXB0b21zID0gc2V0KGxhYmVsLmdldCgic3ltcHRvbSIsIFtdKSkKICAgICAgICBwb3NpdGl2ZXMgKz0gdG9yY2gudGVuc29yKFtpbnQocyBpbiBzeW1wdG9tcykgZm9yIHMgaW4gdGFyZ2V0X3N5bXB0b21zXSkKICAgIG5lZ2F0aXZlcyA9IGxlbihsYWJlbF9maWxlcykgLSBwb3NpdGl2ZXMKICAgICMg7JaR7ISx7J20IOyXhuuKlCDrnbzrsqjsnYAg7LaU7KCVIOu2iOqwgC4g7JaR7ISxIOuLpOyImCDrnbzrsqjrj4Qg7IaQ7Iuk7J2EIOuCruy2lOyngCDslYrripTri6QuCiAgICByYXRpb3MgPSAobmVnYXRpdmVzIC8gcG9zaXRpdmVzLmNsYW1wX21pbigxKSkuY2xhbXBfbWluKDEpCiAgICByYXRpb3MgPSB0b3JjaC53aGVyZShwb3NpdGl2ZXMgPiAwLCByYXRpb3MsIHRvcmNoLm9uZXNfbGlrZShyYXRpb3MpKQogICAgd2VpZ2h0cyA9IDEgKyBzY2FsZSAqIChyYXRpb3MgLSAxKQogICAgaWYgbWF4X3dlaWdodCBpcyBub3QgTm9uZToKICAgICAgICB3ZWlnaHRzID0gd2VpZ2h0cy5jbGFtcF9tYXgobWF4X3dlaWdodCkKICAgIGRldGFpbHMgPSB7CiAgICAgICAgIm5hbWUiOiAiYmNlX3Bvc193ZWlnaHQiLCAic2NhbGUiOiBzY2FsZSwgIm1heF93ZWlnaHQiOiBtYXhfd2VpZ2h0LAogICAgICAgICJ0YXJnZXRfc3ltcHRvbXMiOiBsaXN0KHRhcmdldF9zeW1wdG9tcyksICJudW1fdHJhaW4iOiBsZW4obGFiZWxfZmlsZXMpLAogICAgICAgICJwb3NpdGl2ZV9jb3VudHMiOiBwb3NpdGl2ZXMudG8odG9yY2guaW50NjQpLnRvbGlzdCgpLAogICAgICAgICJuZWdhdGl2ZV9jb3VudHMiOiBuZWdhdGl2ZXMudG8odG9yY2guaW50NjQpLnRvbGlzdCgpLAogICAgICAgICJwb3Nfd2VpZ2h0Ijogd2VpZ2h0cy50b2xpc3QoKSwKICAgIH0KICAgIHJldHVybiB3ZWlnaHRzLCBkZXRhaWxzCg==","metrics_utils.py":"IiIiTWlzc2lvbiAzIG11bHRpLWxhYmVsIO2PieqwgCDsp4DtkZwg7Jyg7Yu466as7YuwLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSB0eXBpbmcgaW1wb3J0IFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKCgpkZWYgX2RpdmlkZShudW1lcmF0b3IsIGRlbm9taW5hdG9yKToKICAgICIiIjDsnLzroZwg64KY64iE64qUIOqyveyasCAw7J2EIOuwmO2ZmO2VmOuKlCDrgpjriJfshYguIiIiCiAgICByZXR1cm4gbnAuZGl2aWRlKAogICAgICAgIG51bWVyYXRvciwKICAgICAgICBkZW5vbWluYXRvciwKICAgICAgICBvdXQ9bnAuemVyb3NfbGlrZShucC5hc2FycmF5KG51bWVyYXRvciwgZHR5cGU9ZmxvYXQpKSwKICAgICAgICB3aGVyZT1ucC5hc2FycmF5KGRlbm9taW5hdG9yKSAhPSAwLAogICAgKQoKCmRlZiBjb21wdXRlX211bHRpbGFiZWxfbWV0cmljcygKICAgIHlfdHJ1ZTogbnAubmRhcnJheSwKICAgIHlfcHJlZDogbnAubmRhcnJheSwKICAgIGxhYmVsX25hbWVzOiBTZXF1ZW5jZVtzdHJdLAopIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgIiIiQmluYXJ5IG11bHRpLWxhYmVsIOyYiOy4oeydmCDsoITssrQv652867Ko67OEIOyngO2RnOulvCDqs4TsgrDtlZzri6QuCgogICAgc2Npa2l0LWxlYXJuIOyXhuydtCBOdW1QeeunjCDsgqzsmqntlbQg7ZWZ7Iq1IO2ZmOqyveydmCDsnZjsobTshLHsnYQg64qY66as7KeAIOyViuuKlOuLpC4KICAgIOuqqOuToCDsnoXroKXsnYAgYGAo7IOY7ZSMIOyImCwg652867KoIOyImClgYCDtmJXtg5zsnZggMC8xIOuwsOyXtOydtOyWtOyVvCDtlZzri6QuCiAgICAiIiIKICAgIHlfdHJ1ZSA9IG5wLmFzYXJyYXkoeV90cnVlLCBkdHlwZT1ucC5pbnQ4KQogICAgeV9wcmVkID0gbnAuYXNhcnJheSh5X3ByZWQsIGR0eXBlPW5wLmludDgpCiAgICBpZiB5X3RydWUuc2hhcGUgIT0geV9wcmVkLnNoYXBlIG9yIHlfdHJ1ZS5uZGltICE9IDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgInlfdHJ1ZeyZgCB5X3ByZWTripQg64+Z7J287ZWcICjsg5jtlIwg7IiYLCDrnbzrsqgg7IiYKSAy7LCo7JuQIOuwsOyXtOydtOyWtOyVvCDtlanri4jri6QuIgogICAgICAgICkKICAgIGlmIHlfdHJ1ZS5zaGFwZVsxXSAhPSBsZW4obGFiZWxfbmFtZXMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhYmVsX25hbWVzIOyImOqwgCDrsLDsl7TsnZgg652867KoIOyImOyZgCDri6TrpoXri4jri6QuIikKICAgIGlmIHlfdHJ1ZS5zaGFwZVswXSA9PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIu2PieqwgO2VoCDsg5jtlIzsnbQg7JeG7Iq164uI64ukLiIpCgogICAgdHAgPSBucC5zdW0oKHlfdHJ1ZSA9PSAxKSAmICh5X3ByZWQgPT0gMSksIGF4aXM9MCkKICAgIGZwID0gbnAuc3VtKCh5X3RydWUgPT0gMCkgJiAoeV9wcmVkID09IDEpLCBheGlzPTApCiAgICBmbiA9IG5wLnN1bSgoeV90cnVlID09IDEpICYgKHlfcHJlZCA9PSAwKSwgYXhpcz0wKQogICAgdG4gPSBucC5zdW0oKHlfdHJ1ZSA9PSAwKSAmICh5X3ByZWQgPT0gMCksIGF4aXM9MCkKCiAgICBwcmVjaXNpb24gPSBfZGl2aWRlKHRwLCB0cCArIGZwKQogICAgcmVjYWxsID0gX2RpdmlkZSh0cCwgdHAgKyBmbikKICAgIGYxID0gX2RpdmlkZSgyICogcHJlY2lzaW9uICogcmVjYWxsLCBwcmVjaXNpb24gKyByZWNhbGwpCiAgICBzcGVjaWZpY2l0eSA9IF9kaXZpZGUodG4sIHRuICsgZnApCiAgICBzdXBwb3J0ID0gdHAgKyBmbgoKICAgIG1pY3JvX3RwID0gaW50KHRwLnN1bSgpKQogICAgbWljcm9fZnAgPSBpbnQoZnAuc3VtKCkpCiAgICBtaWNyb19mbiA9IGludChmbi5zdW0oKSkKICAgIG1pY3JvX3ByZWNpc2lvbiA9IGZsb2F0KF9kaXZpZGUobWljcm9fdHAsIG1pY3JvX3RwICsgbWljcm9fZnApKQogICAgbWljcm9fcmVjYWxsID0gZmxvYXQoX2RpdmlkZShtaWNyb190cCwgbWljcm9fdHAgKyBtaWNyb19mbikpCiAgICBtaWNyb19mMSA9IGZsb2F0KAogICAgICAgIF9kaXZpZGUoCiAgICAgICAgICAgIDIgKiBtaWNyb19wcmVjaXNpb24gKiBtaWNyb19yZWNhbGwsCiAgICAgICAgICAgIG1pY3JvX3ByZWNpc2lvbiArIG1pY3JvX3JlY2FsbCwKICAgICAgICApCiAgICApCgogICAgc2FtcGxlX3RwID0gbnAuc3VtKCh5X3RydWUgPT0gMSkgJiAoeV9wcmVkID09IDEpLCBheGlzPTEpCiAgICBzYW1wbGVfcHJlZGljdGVkID0gbnAuc3VtKHlfcHJlZCA9PSAxLCBheGlzPTEpCiAgICBzYW1wbGVfYWN0dWFsID0gbnAuc3VtKHlfdHJ1ZSA9PSAxLCBheGlzPTEpCiAgICBzYW1wbGVfcHJlY2lzaW9uID0gX2RpdmlkZShzYW1wbGVfdHAsIHNhbXBsZV9wcmVkaWN0ZWQpCiAgICBzYW1wbGVfcmVjYWxsID0gX2RpdmlkZShzYW1wbGVfdHAsIHNhbXBsZV9hY3R1YWwpCiAgICBzYW1wbGVfZjEgPSBfZGl2aWRlKAogICAgICAgIDIgKiBzYW1wbGVfcHJlY2lzaW9uICogc2FtcGxlX3JlY2FsbCwKICAgICAgICBzYW1wbGVfcHJlY2lzaW9uICsgc2FtcGxlX3JlY2FsbCwKICAgICkKCiAgICBzdXBwb3J0X3RvdGFsID0gaW50KHN1cHBvcnQuc3VtKCkpCiAgICB3ZWlnaHRlZF9mMSA9ICgKICAgICAgICBmbG9hdChucC5hdmVyYWdlKGYxLCB3ZWlnaHRzPXN1cHBvcnQpKSBpZiBzdXBwb3J0X3RvdGFsIGVsc2UgMC4wCiAgICApCgogICAgcGVyX2xhYmVsID0gewogICAgICAgIG5hbWU6IHsKICAgICAgICAgICAgInByZWNpc2lvbiI6IGZsb2F0KHByZWNpc2lvbltpbmRleF0pLAogICAgICAgICAgICAicmVjYWxsIjogZmxvYXQocmVjYWxsW2luZGV4XSksCiAgICAgICAgICAgICJmMSI6IGZsb2F0KGYxW2luZGV4XSksCiAgICAgICAgICAgICJzcGVjaWZpY2l0eSI6IGZsb2F0KHNwZWNpZmljaXR5W2luZGV4XSksCiAgICAgICAgICAgICJzdXBwb3J0IjogaW50KHN1cHBvcnRbaW5kZXhdKSwKICAgICAgICAgICAgInRwIjogaW50KHRwW2luZGV4XSksCiAgICAgICAgICAgICJmcCI6IGludChmcFtpbmRleF0pLAogICAgICAgICAgICAiZm4iOiBpbnQoZm5baW5kZXhdKSwKICAgICAgICAgICAgInRuIjogaW50KHRuW2luZGV4XSksCiAgICAgICAgfQogICAgICAgIGZvciBpbmRleCwgbmFtZSBpbiBlbnVtZXJhdGUobGFiZWxfbmFtZXMpCiAgICB9CgogICAgcmV0dXJuIHsKICAgICAgICAibnVtX3NhbXBsZXMiOiBpbnQoeV90cnVlLnNoYXBlWzBdKSwKICAgICAgICAibnVtX2xhYmVscyI6IGludCh5X3RydWUuc2hhcGVbMV0pLAogICAgICAgICJtYWNyb19wcmVjaXNpb24iOiBmbG9hdChucC5tZWFuKHByZWNpc2lvbikpLAogICAgICAgICJtYWNyb19yZWNhbGwiOiBmbG9hdChucC5tZWFuKHJlY2FsbCkpLAogICAgICAgICJtYWNyb19mMSI6IGZsb2F0KG5wLm1lYW4oZjEpKSwKICAgICAgICAibWljcm9fcHJlY2lzaW9uIjogbWljcm9fcHJlY2lzaW9uLAogICAgICAgICJtaWNyb19yZWNhbGwiOiBtaWNyb19yZWNhbGwsCiAgICAgICAgIm1pY3JvX2YxIjogbWljcm9fZjEsCiAgICAgICAgIndlaWdodGVkX2YxIjogd2VpZ2h0ZWRfZjEsCiAgICAgICAgInNhbXBsZXNfcHJlY2lzaW9uIjogZmxvYXQobnAubWVhbihzYW1wbGVfcHJlY2lzaW9uKSksCiAgICAgICAgInNhbXBsZXNfcmVjYWxsIjogZmxvYXQobnAubWVhbihzYW1wbGVfcmVjYWxsKSksCiAgICAgICAgInNhbXBsZXNfZjEiOiBmbG9hdChucC5tZWFuKHNhbXBsZV9mMSkpLAogICAgICAgICJzdWJzZXRfYWNjdXJhY3kiOiBmbG9hdChucC5tZWFuKG5wLmFsbCh5X3RydWUgPT0geV9wcmVkLCBheGlzPTEpKSksCiAgICAgICAgImhhbW1pbmdfbG9zcyI6IGZsb2F0KG5wLm1lYW4oeV90cnVlICE9IHlfcHJlZCkpLAogICAgICAgICJsYWJlbF9hY2N1cmFjeSI6IGZsb2F0KG5wLm1lYW4oeV90cnVlID09IHlfcHJlZCkpLAogICAgICAgICJwZXJfbGFiZWwiOiBwZXJfbGFiZWwsCiAgICB9CgoKZGVmIHByaW50X21ldHJpY3MobWV0cmljczogZGljdFtzdHIsIG9iamVjdF0pIC0+IE5vbmU6CiAgICAiIiLso7zsmpQg7KeA7ZGc7JmAIOudvOuyqOuzhCDsp4DtkZzrpbwg7L2Y7IaU7JeQIOy2nOugpS4iIiIKICAgIHByaW50KCJcblvsoITssrQg7KeA7ZGcXSIpCiAgICBrZXlzID0gWwogICAgICAgICJtYWNyb19mMSIsCiAgICAgICAgIm1pY3JvX2YxIiwKICAgICAgICAid2VpZ2h0ZWRfZjEiLAogICAgICAgICJzYW1wbGVzX2YxIiwKICAgICAgICAibWFjcm9fcHJlY2lzaW9uIiwKICAgICAgICAibWFjcm9fcmVjYWxsIiwKICAgICAgICAic3Vic2V0X2FjY3VyYWN5IiwKICAgICAgICAiaGFtbWluZ19sb3NzIiwKICAgICAgICAibGFiZWxfYWNjdXJhY3kiLAogICAgXQogICAgZm9yIGtleSBpbiBrZXlzOgogICAgICAgIHByaW50KGYiICB7a2V5OjE4c306IHttZXRyaWNzW2tleV06LjRmfSIpCgogICAgcHJpbnQoIlxuW+udvOuyqOuzhCDsp4DtkZxdIikKICAgIHByaW50KGYiICB7J2xhYmVsJzo4c30geydwcmVjaXNpb24nOj45c30geydyZWNhbGwnOj45c30geydmMSc6PjlzfSB7J3N1cHBvcnQnOj45c30iKQogICAgZm9yIGxhYmVsLCB2YWx1ZXMgaW4gbWV0cmljc1sicGVyX2xhYmVsIl0uaXRlbXMoKToKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiIgIHtsYWJlbDo4c30ge3ZhbHVlc1sncHJlY2lzaW9uJ106OS40Zn0gIgogICAgICAgICAgICBmInt2YWx1ZXNbJ3JlY2FsbCddOjkuNGZ9IHt2YWx1ZXNbJ2YxJ106OS40Zn0gIgogICAgICAgICAgICBmInt2YWx1ZXNbJ3N1cHBvcnQnXTo5ZH0iCiAgICAgICAgKQo=","mine_hard_negatives.py":"IiIiVHJhaW5pbmcg64K067aAIO2VmeyKtSBzcGxpdOyXkOyEnCDrqqjrjbggRlAgaGFyZC1uZWdhdGl2ZSDtm4Trs7Trpbwg7LGE6rW07ZWc64ukLgoK6rO17IudIFZhbGlkYXRpb27snYAg7J297KeAIOyViuuKlOuLpC4gYGB0cmFpbi5weWBg7JmAIOuPmeydvO2VnCBgYHNhbXBsZV9yYXRpb2BgLCBgYHNlZWRgYOulvArsgqzsmqntlbQgVHJhaW5pbmcg64K067aAIO2VmeyKtS/qsoDspp0g67aE7ZWg7J2EIOyerO2YhO2VmOqzoCDtlZnsirUg6rWs6rCE66eMIOy2lOuhoO2VnOuLpC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmZyb20gdHFkbSBpbXBvcnQgdHFkbQoKZnJvbSBjb25maWcgaW1wb3J0IERBVEFfUk9PVCwgREVGQVVMVF9DSEVDS1BPSU5UX1BBVEgsIE9VVFBVVF9ESVIsIFRBUkdFVF9TWU1QVE9NUywgZ2V0X3RyYWluX2xhYmVsX2Rpcgpmcm9tIGRhdGFfdXRpbHMgaW1wb3J0IGxpc3RfbGFiZWxfZmlsZXMsIGxvYWRfbGFiZWwKZnJvbSBpbmZlcmVuY2UgaW1wb3J0IGxvYWRfbW9kZWwKZnJvbSBtb2RlbF9taXNzaW9uMyBpbXBvcnQgcG9vbF9jaHVua19sb2dpdHMKZnJvbSB0cmFpbiBpbXBvcnQgTWlzc2lvbjNEYXRhc2V0LCBtYWtlX2xvYWRlciwgc2VsZWN0X3RyYWluaW5nX3N1YnNldCwgc3BsaXRfZmlsZXMKZnJvbSB0cmFuc2NyaXB0X2V2aWRlbmNlIGltcG9ydCBoYXNfY2xlYXJfZXZpZGVuY2UKCgpERUZBVUxUX0ZPQ1VTX1NZTVBUT01TID0gWyLsmKTsi6wiLCAi7KCE7Iug7Ieg7JW9IiwgIu2YuO2doeqzpOuegCJdCgoKZGVmIHNlbGVjdF9oYXJkX25lZ2F0aXZlX3Jvd3MoCiAgICBmaWxlX25hbWVzOiBsaXN0W3N0cl0sCiAgICBsYWJlbHMsCiAgICBwcm9iYWJpbGl0aWVzLAogICAgbGFiZWxfbmFtZXM6IGxpc3Rbc3RyXSB8IHR1cGxlW3N0ciwgLi4uXSwKICAgIGZvY3VzX3N5bXB0b21zOiBsaXN0W3N0cl0gfCB0dXBsZVtzdHIsIC4uLl0sCiAgICB0aHJlc2hvbGQ6IGZsb2F0LAopIC0+IGxpc3RbZGljdF06CiAgICAiIiJiYXRjaOydmCDsnYzshLEg7KCV64u1IOykkSDsnoTqs4TqsJIg7J207IOB7Jy866GcIOyYiOy4oeuQnCDtjIzsnbwt7Kad7IOBIEZQ66W8IOuwmO2ZmC4iIiIKICAgIGZvY3VzX3NldCA9IHNldChmb2N1c19zeW1wdG9tcykKICAgIHJvd3MgPSBbXQogICAgZm9yIHNhbXBsZV9pbmRleCwgZmlsZV9uYW1lIGluIGVudW1lcmF0ZShmaWxlX25hbWVzKToKICAgICAgICBmb3Igc3ltcHRvbV9pbmRleCwgc3ltcHRvbSBpbiBlbnVtZXJhdGUobGFiZWxfbmFtZXMpOgogICAgICAgICAgICB0cnV0aCA9IGludChsYWJlbHNbc2FtcGxlX2luZGV4XVtzeW1wdG9tX2luZGV4XSkKICAgICAgICAgICAgcHJvYmFiaWxpdHkgPSBmbG9hdChwcm9iYWJpbGl0aWVzW3NhbXBsZV9pbmRleF1bc3ltcHRvbV9pbmRleF0pCiAgICAgICAgICAgIGlmIHN5bXB0b20gbm90IGluIGZvY3VzX3NldCBvciB0cnV0aCAhPSAwIG9yIHByb2JhYmlsaXR5IDwgdGhyZXNob2xkOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImZpbGVfbmFtZSI6IGZpbGVfbmFtZSwKICAgICAgICAgICAgICAgICAgICAic3ltcHRvbSI6IHN5bXB0b20sCiAgICAgICAgICAgICAgICAgICAgInRydXRoIjogMCwKICAgICAgICAgICAgICAgICAgICAicHJlZGljdGlvbiI6IDEsCiAgICAgICAgICAgICAgICAgICAgInByb2JhYmlsaXR5IjogcHJvYmFiaWxpdHksCiAgICAgICAgICAgICAgICAgICAgInRocmVzaG9sZCI6IHRocmVzaG9sZCwKICAgICAgICAgICAgICAgICAgICAiZXJyb3JfdHlwZSI6ICJGUCIsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgIHJldHVybiByb3dzCgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgbWluZV9jYW5kaWRhdGVzKAogICAgbW9kZWwsCiAgICBsb2FkZXIsCiAgICB0cmFpbl9maWxlX21hcDogZGljdFtzdHIsIFBhdGhdLAogICAgZGV2aWNlOiB0b3JjaC5kZXZpY2UsCiAgICBmb2N1c19zeW1wdG9tczogbGlzdFtzdHJdLAogICAgdGhyZXNob2xkOiBmbG9hdCwKICAgIHJlcXVpcmVfbm9fY2xlYXJfZXZpZGVuY2U6IGJvb2wsCikgLT4gdHVwbGVbbGlzdFtkaWN0XSwgZGljdF06CiAgICAiIiLrqqjrjbggRlDrpbwg7LC+6rOgIOuqhe2Zle2VnCDsoITsgqwg64uo7ISc6rCAIOyeiOuKlCDrnbzrsqgg64iE6529IOydmOyLrCDsgqzroYDrpbwg7KCc7Jm4LiIiIgogICAgYWNjZXB0ZWQgPSBbXQogICAgZXhjbHVkZWRfY2xlYXJfZXZpZGVuY2UgPSBbXQogICAgZm9yIGJhdGNoIGluIHRxZG0obG9hZGVyLCBkZXNjPSJtaW5lIGhhcmQgbmVnYXRpdmVzIik6CiAgICAgICAgbGFiZWxzX3RlbnNvciA9IGJhdGNoWyJsYWJlbHMiXQogICAgICAgIGNodW5rX2xvZ2l0cyA9IG1vZGVsKAogICAgICAgICAgICBiYXRjaFsiaW5wdXRfaWRzIl0udG8oZGV2aWNlKSwKICAgICAgICAgICAgYmF0Y2hbImF0dGVudGlvbl9tYXNrIl0udG8oZGV2aWNlKSwKICAgICAgICApCiAgICAgICAgbG9naXRzID0gcG9vbF9jaHVua19sb2dpdHMoCiAgICAgICAgICAgIGNodW5rX2xvZ2l0cywKICAgICAgICAgICAgYmF0Y2hbImNvbnZlcnNhdGlvbl9pZHMiXS50byhkZXZpY2UpLAogICAgICAgICAgICBsYWJlbHNfdGVuc29yLnNoYXBlWzBdLAogICAgICAgICkKICAgICAgICBwcm9iYWJpbGl0aWVzID0gdG9yY2guc2lnbW9pZChsb2dpdHMpLmNwdSgpLm51bXB5KCkKICAgICAgICBsYWJlbHMgPSBsYWJlbHNfdGVuc29yLmNwdSgpLm51bXB5KCkKICAgICAgICBjYW5kaWRhdGVzID0gc2VsZWN0X2hhcmRfbmVnYXRpdmVfcm93cygKICAgICAgICAgICAgYmF0Y2hbImZpbGVfbmFtZXMiXSwKICAgICAgICAgICAgbGFiZWxzLAogICAgICAgICAgICBwcm9iYWJpbGl0aWVzLAogICAgICAgICAgICBUQVJHRVRfU1lNUFRPTVMsCiAgICAgICAgICAgIGZvY3VzX3N5bXB0b21zLAogICAgICAgICAgICB0aHJlc2hvbGQsCiAgICAgICAgKQogICAgICAgIGZvciByb3cgaW4gY2FuZGlkYXRlczoKICAgICAgICAgICAgbGFiZWwgPSBsb2FkX2xhYmVsKHRyYWluX2ZpbGVfbWFwW3Jvd1siZmlsZV9uYW1lIl1dKQogICAgICAgICAgICBjbGVhcl9ldmlkZW5jZSA9IGhhc19jbGVhcl9ldmlkZW5jZSgKICAgICAgICAgICAgICAgIGxhYmVsLmdldCgidXR0ZXJhbmNlcyIsIFtdKSwgcm93WyJzeW1wdG9tIl0KICAgICAgICAgICAgKQogICAgICAgICAgICByb3dbImV2aWRlbmNlX2J1Y2tldCJdID0gKAogICAgICAgICAgICAgICAgIuuqhe2Zle2VnCDri6jshJwg7J6I7J2MIiBpZiBjbGVhcl9ldmlkZW5jZSBlbHNlICLri6jshJwg67aA7KGxIOuYkOuKlCDrqqjtmLgiCiAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgY2xlYXJfZXZpZGVuY2UgYW5kIHJlcXVpcmVfbm9fY2xlYXJfZXZpZGVuY2U6CiAgICAgICAgICAgICAgICBleGNsdWRlZF9jbGVhcl9ldmlkZW5jZS5hcHBlbmQocm93KQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgYWNjZXB0ZWQuYXBwZW5kKHJvdykKICAgIGRldGFpbHMgPSB7CiAgICAgICAgIm51bV9tb2RlbF9mcF9wYWlycyI6IGxlbihhY2NlcHRlZCkgKyBsZW4oZXhjbHVkZWRfY2xlYXJfZXZpZGVuY2UpLAogICAgICAgICJudW1fZXhjbHVkZWRfY2xlYXJfZXZpZGVuY2VfcGFpcnMiOiBsZW4oZXhjbHVkZWRfY2xlYXJfZXZpZGVuY2UpLAogICAgICAgICJudW1faGFyZF9uZWdhdGl2ZV9wYWlycyI6IGxlbihhY2NlcHRlZCksCiAgICAgICAgIm51bV9oYXJkX25lZ2F0aXZlX2NvbnZlcnNhdGlvbnMiOiBsZW4oCiAgICAgICAgICAgIHtyb3dbImZpbGVfbmFtZSJdIGZvciByb3cgaW4gYWNjZXB0ZWR9CiAgICAgICAgKSwKICAgIH0KICAgIHJldHVybiBhY2NlcHRlZCwgZGV0YWlscwoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGFfcm9vdCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1EQVRBX1JPT1QpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNrcHRfcGF0aCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1ERUZBVUxUX0NIRUNLUE9JTlRfUEFUSCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tb3V0cHV0IiwKICAgICAgICB0eXBlPVBhdGgsCiAgICAgICAgZGVmYXVsdD1PVVRQVVRfRElSIC8gImhhcmRfbmVnYXRpdmVfbWluaW5nIiAvICJ0cmFpbl9oYXJkX25lZ2F0aXZlcy5jc3YiLAogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaF9zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdGhyZXNob2xkIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNhbXBsZV9yYXRpbyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbWF4X3NhbXBsZXMiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLWZvY3VzX3N5bXB0b21zIiwKICAgICAgICBuYXJncz0iKyIsCiAgICAgICAgZGVmYXVsdD1ERUZBVUxUX0ZPQ1VTX1NZTVBUT01TLAogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1hbGxvd19jbGVhcl9ldmlkZW5jZSIsCiAgICAgICAgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICBoZWxwPSLsoITsgqzsl5Ag66qF7ZmV7ZWcIOyWkeyEsSDri6jshJzqsIAg7J6I64qUIOudvOuyqCDriITrnb0g7J2Y7IusIEZQ64+EIO2PrO2VqCIsCiAgICApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heF9sZW5ndGgiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jaHVua19vdmVybGFwIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgaWYgYXJncy5iYXRjaF9zaXplIDwgMToKICAgICAgICBwYXJzZXIuZXJyb3IoImJhdGNoX3NpemXripQgMSDsnbTsg4HsnbTslrTslbwg7ZWp64uI64ukLiIpCiAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShhcmdzLnRocmVzaG9sZCkgb3Igbm90IDAuMCA8PSBhcmdzLnRocmVzaG9sZCA8PSAxLjA6CiAgICAgICAgcGFyc2VyLmVycm9yKCJ0aHJlc2hvbGTripQg7Jyg7ZWc7ZWcIDB+MSDqsJLsnbTslrTslbwg7ZWp64uI64ukLiIpCiAgICBpZiBhcmdzLnNhbXBsZV9yYXRpbyBpcyBub3QgTm9uZSBhbmQgYXJncy5tYXhfc2FtcGxlcyBpcyBub3QgTm9uZToKICAgICAgICBwYXJzZXIuZXJyb3IoInNhbXBsZV9yYXRpb+yZgCBtYXhfc2FtcGxlc+uKlCDrj5nsi5zsl5Ag7IKs7Jqp7ZWgIOyImCDsl4bsirXri4jri6QuIikKICAgIHVua25vd24gPSBzb3J0ZWQoc2V0KGFyZ3MuZm9jdXNfc3ltcHRvbXMpIC0gc2V0KFRBUkdFVF9TWU1QVE9NUykpCiAgICBpZiB1bmtub3duOgogICAgICAgIHBhcnNlci5lcnJvcihmIuyVjCDsiJgg7JeG64qUIGZvY3VzX3N5bXB0b21zOiB7dW5rbm93bn0iKQogICAgaWYgbm90IGFyZ3MuY2twdF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICBwYXJzZXIuZXJyb3IoZiLssrTtgaztj6zsnbjtirjqsIAg7JeG7Iq164uI64ukOiB7YXJncy5ja3B0X3BhdGh9IikKCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1vZGVsLCB0b2tlbml6ZXIsIGNoZWNrcG9pbnQgPSBsb2FkX21vZGVsKGFyZ3MuY2twdF9wYXRoLCBkZXZpY2UpCiAgICBtYXhfbGVuZ3RoID0gYXJncy5tYXhfbGVuZ3RoIG9yIGNoZWNrcG9pbnQuZ2V0KCJtYXhfbGVuZ3RoIiwgMjU2KQogICAgY2h1bmtfb3ZlcmxhcCA9ICgKICAgICAgICBhcmdzLmNodW5rX292ZXJsYXAKICAgICAgICBpZiBhcmdzLmNodW5rX292ZXJsYXAgaXMgbm90IE5vbmUKICAgICAgICBlbHNlIGNoZWNrcG9pbnQuZ2V0KCJjaHVua19vdmVybGFwIiwgNjQpCiAgICApCiAgICBpZiBtYXhfbGVuZ3RoID4gdG9rZW5pemVyLm1vZGVsX21heF9sZW5ndGg6CiAgICAgICAgcGFyc2VyLmVycm9yKAogICAgICAgICAgICBmIm1heF9sZW5ndGg9e21heF9sZW5ndGh96rCAIHRva2VuaXplciDsp4Dsm5Ag6ri47J20ICIKICAgICAgICAgICAgZiJ7dG9rZW5pemVyLm1vZGVsX21heF9sZW5ndGh966W8IOy0iOqzvO2VqeuLiOuLpC4iCiAgICAgICAgKQoKICAgIGFsbF9maWxlcyA9IGxpc3RfbGFiZWxfZmlsZXMoZ2V0X3RyYWluX2xhYmVsX2RpcihhcmdzLmRhdGFfcm9vdCkpCiAgICBzZWxlY3RlZF9maWxlcyA9IHNlbGVjdF90cmFpbmluZ19zdWJzZXQoCiAgICAgICAgYWxsX2ZpbGVzLAogICAgICAgIHNhbXBsZV9yYXRpbz1hcmdzLnNhbXBsZV9yYXRpbywKICAgICAgICBtYXhfc2FtcGxlcz1hcmdzLm1heF9zYW1wbGVzLAogICAgICAgIHNlZWQ9YXJncy5zZWVkLAogICAgKQogICAgdHJhaW5fZmlsZXMsIHZhbF9maWxlcyA9IHNwbGl0X2ZpbGVzKHNlbGVjdGVkX2ZpbGVzLCBzZWVkPWFyZ3Muc2VlZCkKICAgIHRyYWluX2ZpbGVfbWFwID0ge3BhdGgubmFtZTogcGF0aCBmb3IgcGF0aCBpbiB0cmFpbl9maWxlc30KICAgIGxvYWRlciA9IG1ha2VfbG9hZGVyKAogICAgICAgIE1pc3Npb24zRGF0YXNldCgKICAgICAgICAgICAgdHJhaW5fZmlsZXMsCiAgICAgICAgICAgIHRva2VuaXplciwKICAgICAgICAgICAgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLAogICAgICAgICAgICBjaHVua19vdmVybGFwPWNodW5rX292ZXJsYXAsCiAgICAgICAgKSwKICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICBzaHVmZmxlPUZhbHNlLAogICAgICAgIHNlZWQ9YXJncy5zZWVkLAogICAgKQoKICAgIHJvd3MsIGRldGFpbHMgPSBtaW5lX2NhbmRpZGF0ZXMoCiAgICAgICAgbW9kZWwsCiAgICAgICAgbG9hZGVyLAogICAgICAgIHRyYWluX2ZpbGVfbWFwLAogICAgICAgIGRldmljZSwKICAgICAgICBsaXN0KGRpY3QuZnJvbWtleXMoYXJncy5mb2N1c19zeW1wdG9tcykpLAogICAgICAgIGFyZ3MudGhyZXNob2xkLAogICAgICAgIHJlcXVpcmVfbm9fY2xlYXJfZXZpZGVuY2U9bm90IGFyZ3MuYWxsb3dfY2xlYXJfZXZpZGVuY2UsCiAgICApCiAgICByb3dzLnNvcnQoa2V5PWxhbWJkYSByb3c6IChyb3dbInN5bXB0b20iXSwgLXJvd1sicHJvYmFiaWxpdHkiXSwgcm93WyJmaWxlX25hbWUiXSkpCiAgICBvdXRwdXQgPSBhcmdzLm91dHB1dAogICAgb3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb2x1bW5zID0gWwogICAgICAgICJmaWxlX25hbWUiLAogICAgICAgICJzeW1wdG9tIiwKICAgICAgICAidHJ1dGgiLAogICAgICAgICJwcmVkaWN0aW9uIiwKICAgICAgICAicHJvYmFiaWxpdHkiLAogICAgICAgICJ0aHJlc2hvbGQiLAogICAgICAgICJlcnJvcl90eXBlIiwKICAgICAgICAiZXZpZGVuY2VfYnVja2V0IiwKICAgIF0KICAgIHBkLkRhdGFGcmFtZShyb3dzLCBjb2x1bW5zPWNvbHVtbnMpLnRvX2NzdigKICAgICAgICBvdXRwdXQsIGluZGV4PUZhbHNlLCBlbmNvZGluZz0idXRmLTgtc2lnIgogICAgKQogICAgY291bnRzID0gQ291bnRlcihyb3dbInN5bXB0b20iXSBmb3Igcm93IGluIHJvd3MpCiAgICBzdW1tYXJ5ID0gewogICAgICAgICJjaGVja3BvaW50Ijogc3RyKGFyZ3MuY2twdF9wYXRoLnJlc29sdmUoKSksCiAgICAgICAgIm91dHB1dCI6IHN0cihvdXRwdXQucmVzb2x2ZSgpKSwKICAgICAgICAiZGV2aWNlIjogc3RyKGRldmljZSksCiAgICAgICAgInNlZWQiOiBhcmdzLnNlZWQsCiAgICAgICAgInNhbXBsZV9yYXRpbyI6IGFyZ3Muc2FtcGxlX3JhdGlvLAogICAgICAgICJtYXhfc2FtcGxlcyI6IGFyZ3MubWF4X3NhbXBsZXMsCiAgICAgICAgIm51bV9hbGxfdHJhaW5pbmdfZmlsZXMiOiBsZW4oYWxsX2ZpbGVzKSwKICAgICAgICAibnVtX3NlbGVjdGVkX2ZpbGVzIjogbGVuKHNlbGVjdGVkX2ZpbGVzKSwKICAgICAgICAibnVtX3RyYWluX3NwbGl0X2ZpbGVzIjogbGVuKHRyYWluX2ZpbGVzKSwKICAgICAgICAibnVtX2ludGVybmFsX3ZhbGlkYXRpb25fZmlsZXMiOiBsZW4odmFsX2ZpbGVzKSwKICAgICAgICAiZm9jdXNfc3ltcHRvbXMiOiBsaXN0KGRpY3QuZnJvbWtleXMoYXJncy5mb2N1c19zeW1wdG9tcykpLAogICAgICAgICJ0aHJlc2hvbGQiOiBhcmdzLnRocmVzaG9sZCwKICAgICAgICAicmVxdWlyZV9ub19jbGVhcl9ldmlkZW5jZSI6IG5vdCBhcmdzLmFsbG93X2NsZWFyX2V2aWRlbmNlLAogICAgICAgICoqZGV0YWlscywKICAgICAgICAiaGFyZF9uZWdhdGl2ZV9wYWlyc19ieV9zeW1wdG9tIjogewogICAgICAgICAgICBzeW1wdG9tOiBpbnQoY291bnRzLmdldChzeW1wdG9tLCAwKSkgZm9yIHN5bXB0b20gaW4gYXJncy5mb2N1c19zeW1wdG9tcwogICAgICAgIH0sCiAgICAgICAgImxlYWthZ2VfZ3VhcmQiOiAi6rO17IudIFZhbGlkYXRpb24g66+47IKs7JqpOyDrj5nsnbwgc2VlZOydmCBUcmFpbmluZyDrgrTrtoAgdHJhaW4gc3BsaXTrp4wg7LaU66GgIiwKICAgIH0KICAgIHN1bW1hcnlfcGF0aCA9IG91dHB1dC53aXRoX3N1ZmZpeCgiLnN1bW1hcnkuanNvbiIpCiAgICBzdW1tYXJ5X3BhdGgud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKHN1bW1hcnksIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICBwcmludChqc29uLmR1bXBzKHN1bW1hcnksIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpKQogICAgcHJpbnQoZiJIYXJkLW5lZ2F0aXZlIG1hbmlmZXN0OiB7b3V0cHV0LnJlc29sdmUoKX0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkKCg==","model_mission3.py":"IiIiTWlzc2lvbiAzOiBLY0JFUlQg6riw67CYIO2ZmOyekCDspp3sg4EgbXVsdGktbGFiZWwg67aE66WYIOuqqOuNuC4iIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbAoKCmNsYXNzIFN5bXB0b21DbGFzc2lmaWVyKG5uLk1vZHVsZSk6CiAgICAiIiLrjIDtmZQg7KCE7IKs66Gc67aA7YSwIDnsooUg7Kad7IOB7J2YIGxvZ2l07J2EIOy2nOugpS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWxfbmFtZTogc3RyLCBudW1fbGFiZWxzOiBpbnQpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYmVydCA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQobW9kZWxfbmFtZSkKICAgICAgICBoaWRkZW5fc2l6ZSA9IHNlbGYuYmVydC5jb25maWcuaGlkZGVuX3NpemUKICAgICAgICBzZWxmLmRyb3BvdXQgPSBubi5Ecm9wb3V0KDAuMSkKICAgICAgICBzZWxmLmNsYXNzaWZpZXIgPSBubi5MaW5lYXIoaGlkZGVuX3NpemUsIG51bV9sYWJlbHMpCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICBpbnB1dF9pZHM6IHRvcmNoLlRlbnNvciwKICAgICAgICBhdHRlbnRpb25fbWFzazogdG9yY2guVGVuc29yLAogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgb3V0cHV0cyA9IHNlbGYuYmVydCgKICAgICAgICAgICAgaW5wdXRfaWRzPWlucHV0X2lkcywKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2ssCiAgICAgICAgKQogICAgICAgIHBvb2xlZCA9IHNlbGYuZHJvcG91dChvdXRwdXRzLmxhc3RfaGlkZGVuX3N0YXRlWzosIDBdKQogICAgICAgIHJldHVybiBzZWxmLmNsYXNzaWZpZXIocG9vbGVkKQoKCmRlZiBwb29sX2NodW5rX2xvZ2l0cygKICAgIGNodW5rX2xvZ2l0czogdG9yY2guVGVuc29yLAogICAgY29udmVyc2F0aW9uX2lkczogdG9yY2guVGVuc29yLAogICAgbnVtX2NvbnZlcnNhdGlvbnM6IGludCwKKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiLsl6zrn6wg7LKt7YGs7J2YIGxvZ2l07J2EIOymneyDgeuzhCBtYXggcG9vbGluZ+2VtCDthrXtmZQg64uo7JyE66GcIOuzteybkC4KCiAgICDspp3sg4HsnbQg7Ya17ZmU7J2YIOyWtOuKkCDtlZwg7LKt7YGs7JeQ652864+EIOuCmO2DgOuCmOuptCDrkJzri6TripQgbXVsdGktaW5zdGFuY2UKICAgIGxlYXJuaW5nIOqwgOygleydhCDsgqzsmqntlZzri6QuIHNpZ21vaWTripQg64uo7KGw7ZWo7IiY7J2066+A66GcIGxvZ2l0IG1heCBwb29saW5n7J2ACiAgICDtmZXrpaAgbWF4IHBvb2xpbmfqs7wg6rCZ7J2AIOyyre2BrOulvCDshKDtg53tlZjrqbTshJwg7IiY7LmY7KCB7Jy866GcIOuNlCDslYjsoJXsoIHsnbTri6QuCiAgICAiIiIKICAgIHBvb2xlZCA9IFtdCiAgICBmb3IgY29udmVyc2F0aW9uX2luZGV4IGluIHJhbmdlKG51bV9jb252ZXJzYXRpb25zKToKICAgICAgICBtYXNrID0gY29udmVyc2F0aW9uX2lkcyA9PSBjb252ZXJzYXRpb25faW5kZXgKICAgICAgICBpZiBub3QgdG9yY2guYW55KG1hc2spOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYi7Ya17ZmUIHtjb252ZXJzYXRpb25faW5kZXh97JeQIOuMgOydke2VmOuKlCDssq3tgazqsIAg7JeG7Iq164uI64ukLiIpCiAgICAgICAgcG9vbGVkLmFwcGVuZChjaHVua19sb2dpdHNbbWFza10uYW1heChkaW09MCkpCiAgICByZXR1cm4gdG9yY2guc3RhY2socG9vbGVkLCBkaW09MCkK","run_stratified_role_experiment.py":"...","split_utils.py":"IiIiUHlUb3JjaCDsnZjsobTshLEg7JeG7J20IOyCrOyaqe2VmOuKlCBtdWx0aWxhYmVsIHN0cmF0aWZpZWQgc3BsaXQuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgU2VxdWVuY2UKCmltcG9ydCBudW1weSBhcyBucAoKCmRlZiBpdGVyYXRpdmVfbXVsdGlsYWJlbF9zcGxpdCgKICAgIGl0ZW1zOiBTZXF1ZW5jZSwKICAgIGxhYmVscywKICAgIHZhbF9yYXRpbzogZmxvYXQgPSAwLjEsCiAgICBzZWVkOiBpbnQgPSA0MiwKKToKICAgICIiIu2drOyGjCDrnbzrsqjsnZgg66qp7ZGcIOyImOulvCDsmrDshKAg7LGE7Jqw64qUIOyerO2YhCDqsIDriqXtlZwgOTAvMTAg67aE7ZWgLiIiIgogICAgaXRlbXMgPSBsaXN0KGl0ZW1zKQogICAgeSA9IG5wLmFzYXJyYXkobGFiZWxzLCBkdHlwZT1ucC5pbnQ4KQogICAgaWYgeS5uZGltICE9IDIgb3IgeS5zaGFwZVswXSAhPSBsZW4oaXRlbXMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhYmVsc+uKlCBpdGVtc+yZgCDqsJnsnYAg7ZaJIOyImOydmCAy7LCo7JuQIOuwsOyXtOydtOyWtOyVvCDtlanri4jri6QuIikKICAgIGlmIG5vdCBucC5pc2luKHksIFswLCAxXSkuYWxsKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibGFiZWxz64qUIDAvMeydtOyWtOyVvCDtlanri4jri6QuIikKICAgIGlmIG5vdCAwLjAgPD0gdmFsX3JhdGlvIDwgMS4wOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInZhbF9yYXRpb+uKlCAwIOydtOyDgSAxIOuvuOunjOydtOyWtOyVvCDtlanri4jri6QuIikKICAgIGlmIG5vdCBpdGVtcyBvciB2YWxfcmF0aW8gPT0gMC4wOgogICAgICAgIHJldHVybiBpdGVtcy5jb3B5KCksIFtdCiAgICBpZiBsZW4oaXRlbXMpIDwgMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ2YWxpZGF0aW9uIHNwbGl07JeQ64qUIOy1nOyGjCAy6rCcIOyDmO2UjOydtCDtlYTsmpTtlanri4jri6QuIikKCiAgICB2YWxfc2l6ZSA9IG1pbihsZW4oaXRlbXMpIC0gMSwgbWF4KDEsIHJvdW5kKGxlbihpdGVtcykgKiB2YWxfcmF0aW8pKSkKICAgIGxhYmVsX3RvdGFscyA9IHkuc3VtKGF4aXM9MCkuYXN0eXBlKGZsb2F0KQogICAgZGVzaXJlZCA9IGxhYmVsX3RvdGFscyAqIHZhbF9yYXRpbwogICAgcmFyaXR5ID0gbnAuZGl2aWRlKAogICAgICAgIDEuMCwKICAgICAgICBsYWJlbF90b3RhbHMsCiAgICAgICAgb3V0PW5wLnplcm9zX2xpa2UobGFiZWxfdG90YWxzLCBkdHlwZT1mbG9hdCksCiAgICAgICAgd2hlcmU9bGFiZWxfdG90YWxzID4gMCwKICAgICkKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgdGllX2JyZWFrID0gcm5nLnJhbmRvbShsZW4oaXRlbXMpKQogICAgcmVtYWluaW5nID0gc2V0KHJhbmdlKGxlbihpdGVtcykpKQogICAgc2VsZWN0ZWQ6IGxpc3RbaW50XSA9IFtdCiAgICBjdXJyZW50ID0gbnAuemVyb3MoeS5zaGFwZVsxXSwgZHR5cGU9ZmxvYXQpCgogICAgd2hpbGUgbGVuKHNlbGVjdGVkKSA8IHZhbF9zaXplOgogICAgICAgIG5lZWRzID0gbnAubWF4aW11bShkZXNpcmVkIC0gY3VycmVudCwgMC4wKQoKICAgICAgICBkZWYgY2FuZGlkYXRlX2tleShpbmRleDogaW50KToKICAgICAgICAgICAgcm93ID0geVtpbmRleF0KICAgICAgICAgICAgZ2FpbiA9IGZsb2F0KG5wLnN1bShyb3cgKiBuZWVkcyAqIHJhcml0eSkpCiAgICAgICAgICAgIG92ZXJzaG9vdCA9IGZsb2F0KAogICAgICAgICAgICAgICAgbnAuc3VtKHJvdyAqIG5wLm1heGltdW0oY3VycmVudCArIHJvdyAtIGRlc2lyZWQsIDAuMCkgKiByYXJpdHkpCiAgICAgICAgICAgICkKICAgICAgICAgICAgcmV0dXJuIGdhaW4sIC0wLjIgKiBvdmVyc2hvb3QsIGludChyb3cuc3VtKCkpLCB0aWVfYnJlYWtbaW5kZXhdCgogICAgICAgIGNob3NlbiA9IG1heChyZW1haW5pbmcsIGtleT1jYW5kaWRhdGVfa2V5KQogICAgICAgIHJlbWFpbmluZy5yZW1vdmUoY2hvc2VuKQogICAgICAgIHNlbGVjdGVkLmFwcGVuZChjaG9zZW4pCiAgICAgICAgY3VycmVudCArPSB5W2Nob3Nlbl0KCiAgICBybmcuc2h1ZmZsZShzZWxlY3RlZCkKICAgIHRyYWluX2luZGljZXMgPSBsaXN0KHJlbWFpbmluZykKICAgIHJuZy5zaHVmZmxlKHRyYWluX2luZGljZXMpCiAgICByZXR1cm4gW2l0ZW1zW2ldIGZvciBpIGluIHRyYWluX2luZGljZXNdLCBbaXRlbXNbaV0gZm9yIGkgaW4gc2VsZWN0ZWRdCgo=","threshold_utils.py":"IiIi7ZWZ7Iq17JeQIOyTsOyngCDslYrsnYAg64K067aAIOqygOymneydmCDsoJDsiJjroZzrp4wg7YyQ7KCVIOyehOqzhOqwkuydhCDshKDtg53tlZzri6QuIiIiDQoNCmltcG9ydCBudW1weSBhcyBucA0KDQoNCmRlZiB2YWxpZGF0ZV90aHJlc2hvbGRfZGF0YSh0cnV0aHMsIHNjb3JlcywgbGFiZWxfbmFtZXMpOg0KICAgIHRydXRocywgc2NvcmVzID0gbnAuYXNhcnJheSh0cnV0aHMpLCBucC5hc2FycmF5KHNjb3JlcywgZHR5cGU9ZmxvYXQpDQogICAgaWYgdHJ1dGhzLnNoYXBlICE9IHNjb3Jlcy5zaGFwZSBvciB0cnV0aHMubmRpbSAhPSAyOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLsoJXri7Xqs7wg7KCQ7IiY64qUIOuPmeydvO2VnCAo7Ya17ZmUIOyImCwg7Kad7IOBIOyImCkg67Cw7Je07J207Ja07JW8IO2VqeuLiOuLpC4iKQ0KICAgIGlmIG5vdCBsZW4odHJ1dGhzKSBvciB0cnV0aHMuc2hhcGVbMV0gIT0gbGVuKGxhYmVsX25hbWVzKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi6rKA7KadIO2Gte2ZlOqwgCDsl4bqsbDrgpgg7Kad7IOBIOyInOyEnOydmCDquLjsnbTqsIAg64uk66aF64uI64ukLiIpDQogICAgaWYgbm90IG5wLmlzaW4odHJ1dGhzLCBbMCwgMV0pLmFsbCgpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCLsoJXri7XsnYAgMC8x7J207Ja07JW8IO2VqeuLiOuLpC4iKQ0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShzY29yZXMpLmFsbCgpIG9yIG5wLmFueSgoc2NvcmVzIDwgMCkgfCAoc2NvcmVzID4gMSkpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzaWdtb2lkIOygkOyImOuKlCDsnKDtlZztlZwgMH4xIOqwkuydtOyWtOyVvCDtlanri4jri6QuIikNCiAgICByZXR1cm4gdHJ1dGhzLmFzdHlwZShucC5pbnQ4KSwgc2NvcmVzDQoNCg0KZGVmIHR1bmVfdGhyZXNob2xkcyh0cnV0aHMsIHNjb3JlcywgbGFiZWxfbmFtZXMsIG1ldHJpY19mbiwgb2JqZWN0aXZlPSJtYWNyb19mMSIpOg0KICAgICIiIuqzteyLnSBWYWxpZGF0aW9u7J2AIOydtCDtlajsiJjsl5Ag7KCE64us7ZWY7KeAIOyViuuKlOuLpC4NCg0KICAgIOyghOyXrSDsnoTqs4TqsJLsnYQg7YOQ7IOJ7ZWY6rOgIE1hY3JvIEYxIOuqqeyggeydtOuptCDspp3sg4Hrs4QgRjEg7J6E6rOE6rCS64+EIOu5hOq1kO2VnOuLpC4NCiAgICDrj5nsoJDsnbwg65WM64qUIOq4sOyhtCAwLjXrpbwg7Jqw7ISg7ZWc64ukLiDrgrTrtoAg6rKA7KadIOygkOyImOuKlCDshKDtg53smqkg7KCQ7IiY7J2064ukLg0KICAgICIiIg0KICAgIGlmIG9iamVjdGl2ZSBub3QgaW4gKCJtYWNyb19mMSIsICJzdWJzZXRfYWNjdXJhY3kiKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi66qp7KCBIOyngO2RnOuKlCBtYWNyb19mMSDrmJDripQgc3Vic2V0X2FjY3VyYWN57Jes7JW8IO2VqeuLiOuLpC4iKQ0KICAgIHRydXRocywgc2NvcmVzID0gdmFsaWRhdGVfdGhyZXNob2xkX2RhdGEodHJ1dGhzLCBzY29yZXMsIGxhYmVsX25hbWVzKQ0KICAgIGdyaWQgPSBucC51bmlxdWUobnAucl9bMC4wLCBucC5hcmFuZ2UoMC4wNSwgMC45NTEsIDAuMDEpLCAwLjUsIDAuOTksIDEuMF0pDQogICAgIyDqsJnsnYAg7KeA7ZGc6rCS7J2066m0IOq4sOuzuOqwkuyXkCDqsIDquYzsmrQg7J6E6rOE6rCS7J2EIOyCrOyaqe2VnOuLpC4NCiAgICBncmlkID0gc29ydGVkKGdyaWQudG9saXN0KCksIGtleT1sYW1iZGEgdmFsdWU6IChhYnModmFsdWUgLSAwLjUpLCB2YWx1ZSkpDQoNCiAgICBkZWYgbWV0cmljcyh0aHJlc2hvbGRzKToNCiAgICAgICAgcmV0dXJuIG1ldHJpY19mbih0cnV0aHMsIHNjb3JlcyA+IG5wLmFzYXJyYXkodGhyZXNob2xkcyksIGxhYmVsX25hbWVzKQ0KDQogICAgYmFzZWxpbmUgPSBtZXRyaWNzKDAuNSkNCiAgICBjYW5kaWRhdGVzID0gWygiZml4ZWRfMC41IiwgWzAuNV0gKiBsZW4obGFiZWxfbmFtZXMpLCBiYXNlbGluZSldDQogICAgZ2xvYmFsX3NlYXJjaCA9IFtdDQogICAgYmVzdF9nbG9iYWwsIGdsb2JhbF9tZXRyaWNzID0gMC41LCBiYXNlbGluZQ0KICAgIGZvciB0aHJlc2hvbGQgaW4gZ3JpZDoNCiAgICAgICAgcmVzdWx0ID0gbWV0cmljcyh0aHJlc2hvbGQpDQogICAgICAgIGdsb2JhbF9zZWFyY2guYXBwZW5kKHsidGhyZXNob2xkIjogdGhyZXNob2xkLCAqKnsNCiAgICAgICAgICAgIGtleTogcmVzdWx0W2tleV0gZm9yIGtleSBpbiAoDQogICAgICAgICAgICAgICAgIm1hY3JvX2YxIiwgIm1hY3JvX3ByZWNpc2lvbiIsICJtYWNyb19yZWNhbGwiLA0KICAgICAgICAgICAgICAgICJzdWJzZXRfYWNjdXJhY3kiLCAibGFiZWxfYWNjdXJhY3kiLCAiaGFtbWluZ19sb3NzIiwNCiAgICAgICAgICAgICkNCiAgICAgICAgfX0pDQogICAgICAgIGlmIHJlc3VsdFtvYmplY3RpdmVdID4gZ2xvYmFsX21ldHJpY3Nbb2JqZWN0aXZlXToNCiAgICAgICAgICAgIGJlc3RfZ2xvYmFsLCBnbG9iYWxfbWV0cmljcyA9IHRocmVzaG9sZCwgcmVzdWx0DQogICAgY2FuZGlkYXRlcy5hcHBlbmQoKCJnbG9iYWwiLCBbYmVzdF9nbG9iYWxdICogbGVuKGxhYmVsX25hbWVzKSwgZ2xvYmFsX21ldHJpY3MpKQ0KDQogICAgaWYgb2JqZWN0aXZlID09ICJtYWNyb19mMSI6DQogICAgICAgIHRocmVzaG9sZHMgPSBbMC41XSAqIGxlbihsYWJlbF9uYW1lcykNCiAgICAgICAgZm9yIGluZGV4IGluIHJhbmdlKGxlbihsYWJlbF9uYW1lcykpOg0KICAgICAgICAgICAgdGFyZ2V0ID0gdHJ1dGhzWzosIGluZGV4XQ0KICAgICAgICAgICAgIyDqsoDspp3sl5Ag7ZWc7Kq9IOygleuLteunjCDsnojsnLzrqbQg7ZW064u5IOymneyDgeydmCDsnoTqs4TqsJLsnYAg7LaU7KCV7ZWY7KeAIOyViuuKlOuLpC4NCiAgICAgICAgICAgIGlmIG5vdCB0YXJnZXQuYW55KCkgb3IgdGFyZ2V0LmFsbCgpOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBiZXN0X2YxID0gYmFzZWxpbmVbInBlcl9sYWJlbCJdW2xhYmVsX25hbWVzW2luZGV4XV1bImYxIl0NCiAgICAgICAgICAgIGZvciB0aHJlc2hvbGQgaW4gZ3JpZDoNCiAgICAgICAgICAgICAgICBwcmVkaWN0ZWQgPSBzY29yZXNbOiwgaW5kZXhdID4gdGhyZXNob2xkDQogICAgICAgICAgICAgICAgdHAgPSBucC5zdW0oKHRhcmdldCA9PSAxKSAmIHByZWRpY3RlZCkNCiAgICAgICAgICAgICAgICBmcCA9IG5wLnN1bSgodGFyZ2V0ID09IDApICYgcHJlZGljdGVkKQ0KICAgICAgICAgICAgICAgIGZuID0gbnAuc3VtKCh0YXJnZXQgPT0gMSkgJiB+cHJlZGljdGVkKQ0KICAgICAgICAgICAgICAgIGRlbm9taW5hdG9yID0gMiAqIHRwICsgZnAgKyBmbg0KICAgICAgICAgICAgICAgIGYxID0gMiAqIHRwIC8gZGVub21pbmF0b3IgaWYgZGVub21pbmF0b3IgZWxzZSAwLjANCiAgICAgICAgICAgICAgICBpZiBmMSA+IGJlc3RfZjE6DQogICAgICAgICAgICAgICAgICAgIGJlc3RfZjEsIHRocmVzaG9sZHNbaW5kZXhdID0gZjEsIHRocmVzaG9sZA0KICAgICAgICBjYW5kaWRhdGVzLmFwcGVuZCgoInBlcl9sYWJlbF9mMSIsIHRocmVzaG9sZHMsIG1ldHJpY3ModGhyZXNob2xkcykpKQ0KDQogICAgIyDquLDspIAg7ZuE67O06rCAIOuovOyggOydtOuvgOuhnCDrj5nsoJDsnbTrqbQg6rOg7KCVIDAuNeulvCDsnKDsp4DtlZzri6QuDQogICAgY2hvc2VuID0gbWF4KGNhbmRpZGF0ZXMsIGtleT1sYW1iZGEgY2FuZGlkYXRlOiBjYW5kaWRhdGVbMl1bb2JqZWN0aXZlXSkNCiAgICByZXR1cm4gew0KICAgICAgICAib2JqZWN0aXZlIjogb2JqZWN0aXZlLCAibWV0aG9kIjogY2hvc2VuWzBdLCAidGhyZXNob2xkcyI6IGNob3NlblsxXSwNCiAgICAgICAgInRhcmdldF9zeW1wdG9tcyI6IGxpc3QobGFiZWxfbmFtZXMpLCAibnVtX2ludGVybmFsX3ZhbGlkYXRpb24iOiBsZW4odHJ1dGhzKSwNCiAgICAgICAgInNlbGVjdGlvbl9zb3VyY2UiOiAiaW50ZXJuYWxfdmFsIiwgImludGVybmFsX2NhbmRpZGF0ZXMiOiBbDQogICAgICAgICAgICB7Im1ldGhvZCI6IG1ldGhvZCwgInRocmVzaG9sZHMiOiB0aHJlc2hvbGRzLCAibWV0cmljcyI6IHJlc3VsdH0NCiAgICAgICAgICAgIGZvciBtZXRob2QsIHRocmVzaG9sZHMsIHJlc3VsdCBpbiBjYW5kaWRhdGVzDQogICAgICAgIF0sDQogICAgICAgICJnbG9iYWxfc2VhcmNoIjogc29ydGVkKGdsb2JhbF9zZWFyY2gsIGtleT1sYW1iZGEgcm93OiByb3dbInRocmVzaG9sZCJdKSwNCiAgICB9DQo=","train.py":"...","transcript_evidence.py":"IiIiVHJhaW5pbmcgaGFyZC1uZWdhdGl2ZSDssYTqtbTsl5DshJwg652867KoIOuIhOudvSDsnZjsi6wg7IKs66GA66W8IOygnOyZuO2VmOuKlCDrs7TsiJjsoIEg6rec7LmZLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIEl0ZXJhYmxlCgoKRk9DVVNfRVZJREVOQ0VfUEFUVEVSTlMgPSB7CiAgICAi7Jik7IusIjogWwogICAgICAgIHIi7Jik7IusIiwKICAgICAgICByIuuplOyKpOq7jXzrqZTsiqTqurwiLAogICAgICAgIHIi7Jq466CBKD866rGw66asfOqxsOugpHzqsbDrprwpIiwKICAgICAgICByIu2GoO2VoFxzKig/Ouqyg3zqsbB86rq8KVxzKuqwmSIsCiAgICAgICAgciLqtazsl63sp4h87ZebXHMq6rWs7Jet7KeIIiwKICAgICAgICByIuyGjSg/OuydtHzsnYApP1xzKig/OuuplOyKpXzrr7jsi51866+47IqlfOyauOuggSkiLAogICAgXSwKICAgICLsoITsi6Dsh6Dslb0iOiBbCiAgICAgICAgciLsoITsi6Bccyrsh6Dslb187Ieg7JW9IiwKICAgICAgICByIig/Ouq4sOyatHztnph866elKSg/OuydtHzsnYB864+EKT9ccyooPzrtlZjrgpjrj4RccyopP+yXhiIsCiAgICAgICAgciLstpVccyooPzrripjslrR87LKY7KC4fOyymOyngCkiLAogICAgICAgIHIi66q4KD867J2EfOydtCk/XHMqKD867KCc64yA66GcXHMqKT/rqrtccyrqsIDriIQiLAogICAgICAgIHIi6ry87KedKD867J2EKT9ccyrrqrtccyrtlZgiLAogICAgXSwKICAgICLtmLjtnaHqs6TrnoAiOiBbCiAgICAgICAgciLtmLjtnaFccyrqs6TrnoAiLAogICAgICAgIHIi7IioKD867J2EfOydtCk/XHMqKD867J6YXHMqKT8oPzrrqrtccyrsiax87JWIXHMq7ImsfOyJrOq4sFxzKu2emHzssKh86rCA7IGYfOunie2eiCkiLAogICAgICAgIHIi7Iio7ImsKD866riwfOuKlFxzKuqyjCk/XHMqKD8664SI66y0XHMqKT8oPzrtnpjrk6R87Ja066C1KSIsCiAgICAgICAgciLtmLjtnaEoPzrsnbR87J2AKT9ccyooPzrtnpjrk6R87Ja066C1fOqwgOyBmHzslYhccyrrkJh87J6YXHMq7JWIKSIsCiAgICAgICAgciLsiKjsnYRccyrtl5DrlqEiLAogICAgXSwKfQoKTkVHQVRJT05fUkUgPSByZS5jb21waWxlKAogICAgciIoPzrsl4YoPzrslrR86rOgfOuLpHzripR87Iq164uI64ukfOycvOyLnCk/fOyVhOuLiHzslYp86rSc7LCufOygleyDgXzsoITtmIBccyrslYh866y47KCcXHMq7JeGKSIKKQpISVNUT1JZX1JFID0gcmUuY29tcGlsZSgKICAgIHIiKD867JiI7KCEfOqzvOqxsHzsnpHrhYR87KeA64Kc67KIfO2VnFxzKuyggXzrs5HroKV87Y+J7IaMfOybkOuemHzsnoXsm5Dtloh87Ye07JuQ7ZaIfOyghOyXkFxzKig/Ouq3uOuerHzsnojsl4h87JWTKSkiCikKQUZGSVJNX1JFID0gcmUuY29tcGlsZShyIl5ccyooPzrrhKR87JiIfOunnuyVhOyalHzrp57sirXri4jri6R86re4656Y7JqUfOq3uOugh+yKteuLiOuLpClcYiIpCgoKZGVmIF9tYXRjaGVzX2NsZWFyX2N1cnJlbnQodGV4dDogc3RyLCBzeW1wdG9tOiBzdHIpIC0+IGJvb2w6CiAgICBmb3IgcGF0dGVybiBpbiBGT0NVU19FVklERU5DRV9QQVRURVJOUy5nZXQoc3ltcHRvbSwgW10pOgogICAgICAgIGZvciBtYXRjaCBpbiByZS5maW5kaXRlcihwYXR0ZXJuLCB0ZXh0LCBmbGFncz1yZS5JR05PUkVDQVNFKToKICAgICAgICAgICAgc3RhcnQsIGVuZCA9IG1hdGNoLnNwYW4oKQogICAgICAgICAgICBsb2NhbCA9IHRleHRbbWF4KDAsIHN0YXJ0IC0gMTgpIDogbWluKGxlbih0ZXh0KSwgZW5kICsgMjQpXQogICAgICAgICAgICBwb3NpdGl2ZV9leGNlcHRpb24gPSBib29sKAogICAgICAgICAgICAgICAgcmUuc2VhcmNoKHIiKD867IiofO2YuO2doSkuezAsOH0oPzrrqrt87JWIKS57MCw1fSg/OuyJrHzrkJgpIiwgbG9jYWwpCiAgICAgICAgICAgICkgb3IgKAogICAgICAgICAgICAgICAgc3ltcHRvbSA9PSAi7KCE7Iug7Ieg7JW9IgogICAgICAgICAgICAgICAgYW5kIGJvb2wocmUuc2VhcmNoKHIiKD866riw7Jq0fO2emHzrp6UpLnswLDh97JeGIiwgbG9jYWwpKQogICAgICAgICAgICApCiAgICAgICAgICAgIGlmIE5FR0FUSU9OX1JFLnNlYXJjaChsb2NhbCkgYW5kIG5vdCBwb3NpdGl2ZV9leGNlcHRpb246CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBISVNUT1JZX1JFLnNlYXJjaChsb2NhbCk6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgcmV0dXJuIEZhbHNlCgoKZGVmIGhhc19jbGVhcl9ldmlkZW5jZSgKICAgIHV0dGVyYW5jZXM6IEl0ZXJhYmxlW2RpY3Rbc3RyLCBBbnldXSwKICAgIHN5bXB0b206IHN0ciwKKSAtPiBib29sOgogICAgIiIi7Iug6rOg7J6QIOyngeygkSDsp4TsiKAg65iQ64qUIOyDgeuLtOybkCDsp4jrrLgg65KkIOuqheyLnOyggSDquI3soJUg7J2R64u17J2EIOywvuuKlOuLpC4KCiAgICBzcGVha2VyPTHsnYAg7Iug6rOg7J6QLCBzcGVha2VyPTDsnYAgMTE5IOygkeyImOyekOudvOuKlCDrjbDsnbTthLAg7KCV7J2Y66W8IOyCrOyaqe2VnOuLpC4KICAgIOydtCDtlajsiJjripQg652867Ko7J2EIO2Zleygle2VmOyngCDslYrqs6AgaGFyZC1uZWdhdGl2ZeyXkOyEnCDsoJzsmbjtlaAg67O07IiY7KCBIO2VhO2EsOuLpC4KICAgICIiIgogICAgaXRlbXMgPSBsaXN0KHV0dGVyYW5jZXMpCiAgICBmb3IgaW5kZXgsIHV0dGVyYW5jZSBpbiBlbnVtZXJhdGUoaXRlbXMpOgogICAgICAgIHRleHQgPSBzdHIodXR0ZXJhbmNlLmdldCgidGV4dCIsICIiKSkuc3RyaXAoKQogICAgICAgIGlmIG5vdCB0ZXh0OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHNwZWFrZXIgPSB1dHRlcmFuY2UuZ2V0KCJzcGVha2VyIikKICAgICAgICBpZiBzcGVha2VyID09IDEgYW5kIF9tYXRjaGVzX2NsZWFyX2N1cnJlbnQodGV4dCwgc3ltcHRvbSk6CiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgaWYgc3BlYWtlciAhPSAwIG9yIG5vdCBfbWF0Y2hlc19jbGVhcl9jdXJyZW50KHRleHQsIHN5bXB0b20pOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBuZXh0X3V0dGVyYW5jZSBpbiBpdGVtc1tpbmRleCArIDEgOiBpbmRleCArIDNdOgogICAgICAgICAgICBpZiBuZXh0X3V0dGVyYW5jZS5nZXQoInNwZWFrZXIiKSAhPSAxOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgYW5zd2VyID0gc3RyKG5leHRfdXR0ZXJhbmNlLmdldCgidGV4dCIsICIiKSkuc3RyaXAoKQogICAgICAgICAgICBpZiBBRkZJUk1fUkUuc2VhcmNoKGFuc3dlcikgYW5kIG5vdCBORUdBVElPTl9SRS5zZWFyY2goYW5zd2VyWzozMF0pOgogICAgICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICAgICAgYnJlYWsKICAgIHJldHVybiBGYWxzZQo=","test_experiment_utils.py":"IiIi7IOIIOyLpO2XmOydmCBzcGxpdCwgc2FtcGxlciwg7Jet7ZWgIO2GoO2BsCDrj5nsnpHsnYQg6rKA7Kad7ZWc64ukLiIiIgoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gZGlhbG9ndWVfY2h1bmtpbmcgaW1wb3J0IGJ1aWxkX2RpYWxvZ3VlX3R1cm5zCmZyb20gc3BsaXRfdXRpbHMgaW1wb3J0IGl0ZXJhdGl2ZV9tdWx0aWxhYmVsX3NwbGl0Cgp0cnk6CiAgICBmcm9tIGV4cGVyaW1lbnRfdXRpbHMgaW1wb3J0IEd1YXJhbnRlZWRGdWxsUGFzc1NhbXBsZXIKZXhjZXB0IE1vZHVsZU5vdEZvdW5kRXJyb3I6CiAgICBHdWFyYW50ZWVkRnVsbFBhc3NTYW1wbGVyID0gTm9uZQoKCmNsYXNzIEl0ZXJhdGl2ZVNwbGl0VGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3Rfc3BsaXRfaXNfcmVwcm9kdWNpYmxlX2FuZF9wcmVzZXJ2ZXNfc2l6ZShzZWxmKToKICAgICAgICBpdGVtcyA9IGxpc3QocmFuZ2UoMjApKQogICAgICAgIGxhYmVscyA9IG5wLnplcm9zKCgyMCwgMyksIGR0eXBlPW5wLmludDgpCiAgICAgICAgbGFiZWxzWzoxMCwgMF0gPSAxCiAgICAgICAgbGFiZWxzWzo6NCwgMV0gPSAxCiAgICAgICAgbGFiZWxzW1sxLCA5LCAxN10sIDJdID0gMQogICAgICAgIGZpcnN0ID0gaXRlcmF0aXZlX211bHRpbGFiZWxfc3BsaXQoaXRlbXMsIGxhYmVscywgMC4yLCBzZWVkPTcpCiAgICAgICAgc2Vjb25kID0gaXRlcmF0aXZlX211bHRpbGFiZWxfc3BsaXQoaXRlbXMsIGxhYmVscywgMC4yLCBzZWVkPTcpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChmaXJzdCwgc2Vjb25kKQogICAgICAgIHRyYWluLCB2YWwgPSBmaXJzdAogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHRyYWluKSwgMTYpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4odmFsKSwgNCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNldCh0cmFpbikgfCBzZXQodmFsKSwgc2V0KGl0ZW1zKSkKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHNldCh0cmFpbikgJiBzZXQodmFsKSkKICAgICAgICBzZWxmLmFzc2VydFRydWUobGFiZWxzW3ZhbCwgMl0uYW55KCkpCgogICAgZGVmIHRlc3RfemVyb19yYXRpb19yZXR1cm5zX2Z1bGxfdHJhaW5pbmcoc2VsZik6CiAgICAgICAgdHJhaW4sIHZhbCA9IGl0ZXJhdGl2ZV9tdWx0aWxhYmVsX3NwbGl0KFsxLCAyXSwgW1sxXSwgWzBdXSwgMC4wKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwodHJhaW4sIFsxLCAyXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHZhbCwgW10pCgoKY2xhc3MgR3VhcmFudGVlZFNhbXBsZXJUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBAdW5pdHRlc3Quc2tpcElmKEd1YXJhbnRlZWRGdWxsUGFzc1NhbXBsZXIgaXMgTm9uZSwgIlB5VG9yY2jqsIAg7JeG64qUIOuhnOy7rCDqsoDspp0g7ZmY6rK9IikKICAgIGRlZiB0ZXN0X2V2ZXJ5X2luZGV4X2FwcGVhcnNfYW5kX29ubHlfaGFyZF9uZWdhdGl2ZXNfcmVwZWF0KHNlbGYpOgogICAgICAgIHNhbXBsZXIgPSBHdWFyYW50ZWVkRnVsbFBhc3NTYW1wbGVyKFsxLjAsIDEuNSwgMS4wLCAxLjVdLCBzZWVkPTExKQogICAgICAgIGluZGljZXMgPSBsaXN0KGl0ZXIoc2FtcGxlcikpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oaW5kaWNlcyksIDUpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKHNldChyYW5nZSg0KSkuaXNzdWJzZXQoaW5kaWNlcykpCiAgICAgICAgcmVwZWF0ZWQgPSBbaW5kZXggZm9yIGluZGV4IGluIHNldChpbmRpY2VzKSBpZiBpbmRpY2VzLmNvdW50KGluZGV4KSA+IDFdCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKHNldChyZXBlYXRlZCkuaXNzdWJzZXQoezEsIDN9KSkKCgpjbGFzcyBTcGVha2VyUm9sZVRlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3JvbGVfdG9rZW5zX2FyZV9vcHRpb25hbChzZWxmKToKICAgICAgICB1dHRlcmFuY2VzID0gWwogICAgICAgICAgICB7InNwZWFrZXIiOiAwLCAidGV4dCI6ICLsiKjsiazquLAg7Z6Y65Oc7IS47JqUPyJ9LAogICAgICAgICAgICB7InNwZWFrZXIiOiAxLCAidGV4dCI6ICLslYTri4jsmpQuIn0sCiAgICAgICAgXQogICAgICAgIHBsYWluID0gYnVpbGRfZGlhbG9ndWVfdHVybnModXR0ZXJhbmNlcywgc3BlYWtlcl9yb2xlX3Rva2Vucz1GYWxzZSkKICAgICAgICBtYXJrZWQgPSBidWlsZF9kaWFsb2d1ZV90dXJucyh1dHRlcmFuY2VzLCBzcGVha2VyX3JvbGVfdG9rZW5zPVRydWUpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwbGFpbiwgWyLsiKjsiazquLAg7Z6Y65Oc7IS47JqUPyIsICLslYTri4jsmpQuIl0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChtYXJrZWQsIFsiW+yDgeuLtOybkF0g7Iio7Ims6riwIO2emOuTnOyEuOyalD8iLCAiW+yLoOqzoOyekF0g7JWE64uI7JqULiJdKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB1bml0dGVzdC5tYWluKCkK","test_threshold_utils.py":"IiIi7J6E6rOE6rCSIOyEoO2DneydtCDrgrTrtoAg6rKA7Kad7JeQ7IScIOygleydmO2VnCDrqqnsoIEg7KeA7ZGc66W8IOuUsOultOuKlOyngCDtmZXsnbjtlZzri6QuIiIiCgppbXBvcnQgdW5pdHRlc3QKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBtZXRyaWNzX3V0aWxzIGltcG9ydCBjb21wdXRlX211bHRpbGFiZWxfbWV0cmljcwpmcm9tIHRocmVzaG9sZF91dGlscyBpbXBvcnQgdHVuZV90aHJlc2hvbGRzCgoKY2xhc3MgVGhyZXNob2xkVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3Rfb3ZlcnByZWRpY3Rpb25fY2FuX2JlX2NvcnJlY3RlZF93aXRob3V0X3RyYWluaW5nKHNlbGYpOgogICAgICAgIHRydXRocyA9IG5wLmFycmF5KFtbMSwgMF0sIFswLCAxXSwgWzAsIDBdLCBbMSwgMV1dKQogICAgICAgIHNjb3JlcyA9IG5wLmFycmF5KFtbMC45LCAwLjhdLCBbMC43LCAwLjldLCBbMC42NSwgMC42NV0sIFswLjksIDAuOV1dKQogICAgICAgIHJlc3VsdCA9IHR1bmVfdGhyZXNob2xkcyh0cnV0aHMsIHNjb3JlcywgWyJhIiwgImIiXSwgY29tcHV0ZV9tdWx0aWxhYmVsX21ldHJpY3MpCiAgICAgICAgc2VsZWN0ZWQgPSBjb21wdXRlX211bHRpbGFiZWxfbWV0cmljcyh0cnV0aHMsIHNjb3JlcyA+IHJlc3VsdFsidGhyZXNob2xkcyJdLCBbImEiLCAiYiJdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2VsZWN0ZWRbIm1hY3JvX2YxIl0sIDEuMCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJlc3VsdFsic2VsZWN0aW9uX3NvdXJjZSJdLCAiaW50ZXJuYWxfdmFsIikKCiAgICBkZWYgdGVzdF9zdWJzZXRfb2JqZWN0aXZlX3VzZXNfZ2xvYmFsX2NhbmRpZGF0ZXMoc2VsZik6CiAgICAgICAgdHJ1dGhzID0gbnAuYXJyYXkoW1sxLCAwXSwgWzAsIDFdLCBbMCwgMF0sIFsxLCAxXV0pCiAgICAgICAgc2NvcmVzID0gbnAuYXJyYXkoW1swLjksIDAuOF0sIFswLjcsIDAuOV0sIFswLjY1LCAwLjY1XSwgWzAuOSwgMC45XV0pCiAgICAgICAgcmVzdWx0ID0gdHVuZV90aHJlc2hvbGRzKHRydXRocywgc2NvcmVzLCBbImEiLCAiYiJdLCBjb21wdXRlX211bHRpbGFiZWxfbWV0cmljcywgInN1YnNldF9hY2N1cmFjeSIpCiAgICAgICAgY2FuZGlkYXRlcyA9IHJlc3VsdFsiaW50ZXJuYWxfY2FuZGlkYXRlcyJdCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChbY1sibWV0aG9kIl0gZm9yIGMgaW4gY2FuZGlkYXRlc10sIFsiZml4ZWRfMC41IiwgImdsb2JhbCJdKQogICAgICAgIHNlbGVjdGVkID0gbmV4dChjIGZvciBjIGluIGNhbmRpZGF0ZXMgaWYgY1sibWV0aG9kIl0gPT0gcmVzdWx0WyJtZXRob2QiXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNlbGVjdGVkWyJtZXRyaWNzIl1bInN1YnNldF9hY2N1cmFjeSJdLCAxLjApCgogICAgZGVmIHRlc3RfZXF1YWxfc2NvcmVzX3ByZXNlcnZlX2Jhc2VsaW5lKHNlbGYpOgogICAgICAgIHRydXRocyA9IG5wLmFycmF5KFtbMV0sIFswXV0pCiAgICAgICAgc2NvcmVzID0gbnAuYXJyYXkoW1swLjldLCBbMC4xXV0pCiAgICAgICAgcmVzdWx0ID0gdHVuZV90aHJlc2hvbGRzKHRydXRocywgc2NvcmVzLCBbImEiXSwgY29tcHV0ZV9tdWx0aWxhYmVsX21ldHJpY3MpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbIm1ldGhvZCJdLCAiZml4ZWRfMC41IikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJlc3VsdFsidGhyZXNob2xkcyJdLCBbMC41XSkKCiAgICBkZWYgdGVzdF9taXNzaW5nX2NsYXNzX3ByZXNlcnZlc19wZXJfbGFiZWxfdGhyZXNob2xkKHNlbGYpOgogICAgICAgIHRydXRocyA9IG5wLmFycmF5KFtbMCwgMV0sIFswLCAwXV0pCiAgICAgICAgc2NvcmVzID0gbnAuYXJyYXkoW1swLjcsIDAuOV0sIFswLjcsIDAuN11dKQogICAgICAgIHJlc3VsdCA9IHR1bmVfdGhyZXNob2xkcyh0cnV0aHMsIHNjb3JlcywgWyJhYnNlbnQiLCAicHJlc2VudCJdLCBjb21wdXRlX211bHRpbGFiZWxfbWV0cmljcykKICAgICAgICBwZXJfbGFiZWwgPSBuZXh0KGMgZm9yIGMgaW4gcmVzdWx0WyJpbnRlcm5hbF9jYW5kaWRhdGVzIl0gaWYgY1sibWV0aG9kIl0gPT0gInBlcl9sYWJlbF9mMSIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwZXJfbGFiZWxbInRocmVzaG9sZHMiXVswXSwgMC41KQoKICAgIGRlZiB0ZXN0X2ludmFsaWRfZGF0YV9hbmRfb2JqZWN0aXZlKHNlbGYpOgogICAgICAgIGZvciBzY29yZXMgaW4gKFtbZmxvYXQoIm5hbiIpXV0sIFtbMS41XV0pOgogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICAgICAgdHVuZV90aHJlc2hvbGRzKFtbMV1dLCBzY29yZXMsIFsiYSJdLCBjb21wdXRlX211bHRpbGFiZWxfbWV0cmljcykKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICB0dW5lX3RocmVzaG9sZHMoW1sxXV0sIFtbMC44XV0sIFsiYSJdLCBjb21wdXRlX211bHRpbGFiZWxfbWV0cmljcywgImFjY3VyYWN5IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==","test_turn_chunking.py":"IiIi67Cc7ZmUIOqyveqzhCDssq3tgazsnZgg66y466elIOuztOyhtMK364iE6529wrfquLjsnbQg7KCc7ZWcIO2ajOq3gCDthYzsiqTtirguIiIiDQoNCmltcG9ydCByYW5kb20NCmltcG9ydCB1bml0dGVzdA0KDQpmcm9tIGRpYWxvZ3VlX2NodW5raW5nIGltcG9ydCBidWlsZF9kaWFsb2d1ZV90dXJucywgYnVpbGRfZGlhbG9ndWVfdG9rZW5fY2h1bmtzDQoNCg0KY2xhc3MgRmFrZVRva2VuaXplcjoNCiAgICBkZWYgbnVtX3NwZWNpYWxfdG9rZW5zX3RvX2FkZChzZWxmLCBwYWlyPUZhbHNlKToNCiAgICAgICAgcmV0dXJuIDINCg0KICAgIGRlZiBfX2NhbGxfXyhzZWxmLCB0ZXh0LCAqKmt3YXJncyk6DQogICAgICAgIHJldHVybiB7ImlucHV0X2lkcyI6IFtpbnQodG9rZW4pIGZvciB0b2tlbiBpbiB0ZXh0LnNwbGl0KCldfQ0KDQoNCmRlZiB1dHRlcmFuY2VzX2Zyb21fdHVybnModHVybnMpOg0KICAgIHJldHVybiBbeyJzcGVha2VyIjogaW5kZXggJSAyLCAidGV4dCI6ICIgIi5qb2luKG1hcChzdHIsIHRva2VucykpfQ0KICAgICAgICAgICAgZm9yIGluZGV4LCB0b2tlbnMgaW4gZW51bWVyYXRlKHR1cm5zKV0NCg0KDQpkZWYgY29udGFpbnNfc2VxdWVuY2UoY2h1bmssIHNlcXVlbmNlKToNCiAgICByZXR1cm4gYW55KGNodW5rW3N0YXJ0OnN0YXJ0ICsgbGVuKHNlcXVlbmNlKV0gPT0gc2VxdWVuY2UNCiAgICAgICAgICAgICAgIGZvciBzdGFydCBpbiByYW5nZShsZW4oY2h1bmspIC0gbGVuKHNlcXVlbmNlKSArIDEpKQ0KDQoNCmNsYXNzIFR1cm5DaHVua2luZ1Rlc3QodW5pdHRlc3QuVGVzdENhc2UpOg0KICAgIGRlZiB0ZXN0X3F1ZXN0aW9uX2FuZF9uZXh0X2Fuc3dlcl9zaGFyZV9hX2NodW5rKHNlbGYpOg0KICAgICAgICB0dXJucyA9IFtbMSwgMl0sIFszLCA0XSwgWzUsIDZdXQ0KICAgICAgICBjaHVua3MgPSBidWlsZF9kaWFsb2d1ZV90b2tlbl9jaHVua3MoDQogICAgICAgICAgICBGYWtlVG9rZW5pemVyKCksIHV0dGVyYW5jZXNfZnJvbV90dXJucyh0dXJucyksIG1heF9sZW5ndGg9NywgY2h1bmtfb3ZlcmxhcD0xLA0KICAgICAgICApDQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoY2h1bmtzLCBbWzEsIDIsIDMsIDRdLCBbMywgNCwgNSwgNl1dKQ0KDQogICAgZGVmIHRlc3RfY29uc2VjdXRpdmVfc2FtZV9zcGVha2VyX2lzX29uZV90dXJuKHNlbGYpOg0KICAgICAgICB0dXJucyA9IGJ1aWxkX2RpYWxvZ3VlX3R1cm5zKFsNCiAgICAgICAgICAgIHsic3BlYWtlciI6IDAsICJ0ZXh0IjogIu2GoCJ9LA0KICAgICAgICAgICAgeyJzcGVha2VyIjogMCwgInRleHQiOiAi7ZWY7IWo7Ja07JqUPyJ9LA0KICAgICAgICAgICAgeyJzcGVha2VyIjogMSwgInRleHQiOiAi7JWE64uI7JqULiJ9LA0KICAgICAgICAgICAgeyJzcGVha2VyIjogMSwgInRleHQiOiAi7IaN66eMIOyauOuggeqxsOugpOyalC4ifSwNCiAgICAgICAgXSkNCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCh0dXJucywgWyLthqAg7ZWY7IWo7Ja07JqUPyIsICLslYTri4jsmpQuIOyGjeunjCDsmrjroIHqsbDroKTsmpQuIl0pDQoNCiAgICBkZWYgdGVzdF9taXNzaW5nX3NwZWFrZXJzX2FyZV9ub3RfbWVyZ2VkKHNlbGYpOg0KICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGJ1aWxkX2RpYWxvZ3VlX3R1cm5zKFsNCiAgICAgICAgICAgIHsidGV4dCI6ICLsp4jrrLgifSwgeyJ0ZXh0IjogIuuLteuzgCJ9LA0KICAgICAgICBdKSwgWyLsp4jrrLgiLCAi64u167OAIl0pDQoNCiAgICBkZWYgdGVzdF9vdmVyc2l6ZWRfcGFpcl9oYXNfYV9ib3VuZGFyeV9jb250ZXh0X2NodW5rKHNlbGYpOg0KICAgICAgICBjaHVua3MgPSBidWlsZF9kaWFsb2d1ZV90b2tlbl9jaHVua3MoDQogICAgICAgICAgICBGYWtlVG9rZW5pemVyKCksIHV0dGVyYW5jZXNfZnJvbV90dXJucyhbWzEsIDIsIDMsIDRdLCBbNSwgNiwgNywgOF1dKSwNCiAgICAgICAgICAgIG1heF9sZW5ndGg9NywgY2h1bmtfb3ZlcmxhcD0xLA0KICAgICAgICApDQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbnkoY29udGFpbnNfc2VxdWVuY2UoY2h1bmssIFs0LCA1XSkgZm9yIGNodW5rIGluIGNodW5rcykpDQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwobGVuKGNodW5rKSA8PSA1IGZvciBjaHVuayBpbiBjaHVua3MpKQ0KICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHt0b2tlbiBmb3IgY2h1bmsgaW4gY2h1bmtzIGZvciB0b2tlbiBpbiBjaHVua30sIHNldChyYW5nZSgxLCA5KSkpDQoNCiAgICBkZWYgdGVzdF9sb25nX3R1cm5fdXNlc190b2tlbl9vdmVybGFwX3dpdGhvdXRfZHJvcHBpbmdfdG9rZW5zKHNlbGYpOg0KICAgICAgICBjaHVua3MgPSBidWlsZF9kaWFsb2d1ZV90b2tlbl9jaHVua3MoDQogICAgICAgICAgICBGYWtlVG9rZW5pemVyKCksIHV0dGVyYW5jZXNfZnJvbV90dXJucyhbbGlzdChyYW5nZSgxLCAyMSkpXSksDQogICAgICAgICAgICBtYXhfbGVuZ3RoPTgsIGNodW5rX292ZXJsYXA9MiwNCiAgICAgICAgKQ0KICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGNodW5rcywgW2xpc3QocmFuZ2Uoc3RhcnQsIG1pbihzdGFydCArIDYsIDIxKSkpDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIHN0YXJ0IGluIFsxLCA1LCA5LCAxMywgMTddXSkNCg0KICAgIGRlZiB0ZXN0X3JhbmRvbV9kaWFsb2d1ZXNfcHJlc2VydmVfZXZlcnlfdG9rZW5fYW5kX2ZpdHRpbmdfYWRqYWNlbnRfcGFpcihzZWxmKToNCiAgICAgICAgcm5nID0gcmFuZG9tLlJhbmRvbSg0MikNCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UoMTAwKToNCiAgICAgICAgICAgIHR1cm5zID0gW10NCiAgICAgICAgICAgIG5leHRfdG9rZW4gPSAxMDAwDQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShybmcucmFuZGludCgxLCAyMCkpOg0KICAgICAgICAgICAgICAgIHNpemUgPSBybmcucmFuZGludCgxLCAyNSkNCiAgICAgICAgICAgICAgICB0dXJucy5hcHBlbmQobGlzdChyYW5nZShuZXh0X3Rva2VuLCBuZXh0X3Rva2VuICsgc2l6ZSkpKQ0KICAgICAgICAgICAgICAgIG5leHRfdG9rZW4gKz0gc2l6ZQ0KICAgICAgICAgICAgY2h1bmtzID0gYnVpbGRfZGlhbG9ndWVfdG9rZW5fY2h1bmtzKA0KICAgICAgICAgICAgICAgIEZha2VUb2tlbml6ZXIoKSwgdXR0ZXJhbmNlc19mcm9tX3R1cm5zKHR1cm5zKSwgbWF4X2xlbmd0aD0xOCwgY2h1bmtfb3ZlcmxhcD00LA0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFsbChsZW4oY2h1bmspIDw9IDE2IGZvciBjaHVuayBpbiBjaHVua3MpKQ0KICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCh7dG9rZW4gZm9yIGNodW5rIGluIGNodW5rcyBmb3IgdG9rZW4gaW4gY2h1bmt9LA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICB7dG9rZW4gZm9yIHR1cm4gaW4gdHVybnMgZm9yIHRva2VuIGluIHR1cm59KQ0KICAgICAgICAgICAgZm9yIGZpcnN0LCBzZWNvbmQgaW4gemlwKHR1cm5zLCB0dXJuc1sxOl0pOg0KICAgICAgICAgICAgICAgIGlmIGxlbihmaXJzdCkgKyBsZW4oc2Vjb25kKSA8PSAxNjoNCiAgICAgICAgICAgICAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFueShjb250YWluc19zZXF1ZW5jZShjaHVuaywgZmlyc3QgKyBzZWNvbmQpDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGNodW5rIGluIGNodW5rcykpDQogICAgICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFueShjb250YWluc19zZXF1ZW5jZShjaHVuaywgW2ZpcnN0Wy0xXSwgc2Vjb25kWzBdXSkNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgY2h1bmsgaW4gY2h1bmtzKSkNCg0KDQoNCiAgICBkZWYgdGVzdF9lbXB0eV9kaWFsb2d1ZV9yZXR1cm5zX29uZV9lbXB0eV9jb250ZW50X2NodW5rKHNlbGYpOg0KICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGJ1aWxkX2RpYWxvZ3VlX3Rva2VuX2NodW5rcygNCiAgICAgICAgICAgIEZha2VUb2tlbml6ZXIoKSwgW10sIG1heF9sZW5ndGg9OCwgY2h1bmtfb3ZlcmxhcD0yLA0KICAgICAgICApLCBbW11dKQ0KDQogICAgZGVmIHRlc3RfbWluaW11bV9jYXBhY2l0eV9hbmRfemVyb19vdmVybGFwKHNlbGYpOg0KICAgICAgICBjaHVua3MgPSBidWlsZF9kaWFsb2d1ZV90b2tlbl9jaHVua3MoDQogICAgICAgICAgICBGYWtlVG9rZW5pemVyKCksIHV0dGVyYW5jZXNfZnJvbV90dXJucyhbWzEsIDIsIDNdLCBbNF1dKSwNCiAgICAgICAgICAgIG1heF9sZW5ndGg9MywgY2h1bmtfb3ZlcmxhcD0wLA0KICAgICAgICApDQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoY2h1bmtzLCBbWzFdLCBbMl0sIFszXSwgWzRdXSkNCg0KICAgIGRlZiB0ZXN0X2ludmFsaWRfc2V0dGluZ3Moc2VsZik6DQogICAgICAgIGZvciBtYXhfbGVuZ3RoLCBvdmVybGFwIGluIFsoMiwgMCksICg4LCAtMSksICg4LCA2KV06DQogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOg0KICAgICAgICAgICAgICAgIGJ1aWxkX2RpYWxvZ3VlX3Rva2VuX2NodW5rcyhGYWtlVG9rZW5pemVyKCksIFtdLCBtYXhfbGVuZ3RoLCBvdmVybGFwKQ0KDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQogICAgdW5pdHRlc3QubWFpbigpDQo="}
EMBEDDED_SHA256 = {"config.py":"7d90b6fbb9bccb67ddbdb199b85307d01a2603320d745f08d10843e2f4929a3a","data_utils.py":"49f465a6efa01c5be648bf7b972379e4ff38debf38761267ec4913b4427161ec","dialogue_chunking.py":"4856bc84d61dfe72e802a8cd35ca8650a047c3baff265bacb08cefa2294c02bc","evaluate.py":"c9d8309c2166d88cd870197f524b1f04f358995473b14926d5894200acc7db6f","experiment_utils.py":"87f5c7e0c259ae4b706ab690068248287ec002585883bfbdc0e7d8bffa764b26","hard_negative_utils.py":"2ffbbedc39b494304c529d65d69825547285b5bbe5e725edc393dc8152b5999c","inference.py":"3d6bf227fd61850230961a6b6824683a146ce663b0544506905fae1f9f6f8817","loss_utils.py":"b49da9187760c4ea207f51ef5a3e6f3cf22a203820c442259ca29fbd785fe348","metrics_utils.py":"4c1fcf91958e8bbee4353121c701a1638e88382fc19b20d826d920d13c4cf5b5","mine_hard_negatives.py":"bba31d1e861e2268233c1cb52b936a4333e7dd415e42b26bd20ee7cd942c18fd","model_mission3.py":"c46e87046cb5fd43048a30cb726764b7a640591af7b9a5250fc8a762e87368db","run_stratified_role_experiment.py":"6c5fbba868daf250f1dbd8c7b3bc87ad694d4b16a00fdbcd1d93a7fc2f49c7d8","split_utils.py":"8c6f2e532c7c480d22cdd4d7e7874391e8275388b308806c4c0ff2c559f89bfe","threshold_utils.py":"717d48e605122ac48da31559e03b56421e18b91b806620cd9c0a72628621f7d8","train.py":"2cf0f1aa427741a9e26208a17c754aa3204cd127b4c04301d61729d084746879","transcript_evidence.py":"47e89ed4e93586de9ea659f678bae642214296bccad8faa3d268b7464b80afc1","test_experiment_utils.py":"65219aaaa0e6bde12bc95ece92e590541f6fef737dd7c19ef92a7c7a7aa83a79","test_threshold_utils.py":"999f77390860f3dd3fa8a9b9f26d6aab6e0c503361329f6bab449ac1845fd215","test_turn_chunking.py":"f5da5f50c4b2d9ee6210d47f93bc6b8fe81933dfa36511339b4ca27d76cc8874"}

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. 런타임 유형을 T4 GPU 이상으로 변경하세요.")
if _dataset_layout(DATA_ROOT) is None:
    raise FileNotFoundError("선택된 DATA_ROOT의 Training/Validation 구조를 읽을 수 없습니다: " + str(DATA_ROOT))
if not BASELINE_CHECKPOINT.is_file():
    raise FileNotFoundError("기준 체크포인트가 없습니다: " + str(BASELINE_CHECKPOINT))

LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
for name, payload in EMBEDDED_FILES.items():
    raw = base64.b64decode(payload)
    digest = hashlib.sha256(raw).hexdigest()
    if digest != EMBEDDED_SHA256[name]:
        raise RuntimeError("내장 파일 해시 불일치: " + name)
    (LOCAL_PROJECT_DIR / name).write_bytes(raw)

(LOCAL_PROJECT_DIR / "BUNDLE_VERSION.txt").write_text(BUNDLE_VERSION, encoding="utf-8")
print("GPU:", torch.cuda.get_device_name(0))
print("내장 코드 준비 완료:", BUNDLE_VERSION)
print("파일 수:", len(EMBEDDED_FILES))


## 5. 코드 검사

실제 학습 전에 층화 분할, 전체 노출 sampler, 임계값 선택, 역할 토큰 청크 테스트를 실행합니다.


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "unittest",
    "test_experiment_utils", "test_threshold_utils", "test_turn_chunking", "-v",
], cwd=LOCAL_PROJECT_DIR, check=True)


## 6. 실행 명령 확인

이 셀은 학습하지 않고 명령과 실험 계획만 출력합니다. `오심`이 대상에 없고, 90/10·1.5배·역할 토큰 비교·Macro F1 선택·Training 100% 재학습이 표시되어야 합니다.


In [ ]:
def build_run_command(dry_run=False):
    command = [
        sys.executable, str(LOCAL_PROJECT_DIR / "run_stratified_role_experiment.py"),
        "--data_root", str(DATA_ROOT),
        "--baseline_checkpoint", str(BASELINE_CHECKPOINT),
        "--output_dir", str(DRIVE_RUN_DIR),
        "--model_name", MODEL_NAME,
        "--seed", str(SEED),
        "--internal_val_ratio", str(INTERNAL_VAL_RATIO),
        "--max_epochs", str(MAX_EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
        "--learning_rate", str(LEARNING_RATE),
        "--max_length", str(MAX_LENGTH),
        "--chunk_overlap", str(CHUNK_OVERLAP),
        "--pos_weight_scale", str(POS_WEIGHT_SCALE),
        "--pos_weight_max", str(POS_WEIGHT_MAX),
        "--hard_negative_oversample", str(HARD_NEGATIVE_OVERSAMPLE),
        "--hard_negative_symptoms", *HARD_NEGATIVE_SYMPTOMS,
    ]
    if ALLOW_INTERNAL_THRESHOLD_TUNING:
        command.append("--tune_thresholds")
    if RUN_MODE in ("resume", "evaluate_only"):
        command.append("--resume")
    if dry_run:
        command.append("--dry_run")
    return command


dry_run_result = subprocess.run(
    build_run_command(dry_run=True),
    cwd=LOCAL_PROJECT_DIR,
    text=True,
    capture_output=True,
)
print(dry_run_result.stdout)
if dry_run_result.returncode != 0:
    print(dry_run_result.stderr)
    raise RuntimeError("실행 명령 사전 검사가 실패했습니다.")


## 7. 90/10 비교 실험 및 Training 100% 재학습

출력은 실시간으로 표시됩니다. 오류가 발생하면 마지막 Python 예외가 그대로 보입니다. 재실행 시 split, hard-negative manifest, 완료된 모델 비교 결과, 최종 체크포인트를 순서대로 재사용합니다.


In [ ]:
import gc
import os
import subprocess
import torch

if FINAL_CHECKPOINT.is_file() and EXPERIMENT_SUMMARY.is_file():
    print("최종 재학습 결과가 이미 있어 학습 단계를 건너뜁니다:", FINAL_CHECKPOINT)
else:
    gc.collect()
    torch.cuda.empty_cache()
    child_env = os.environ.copy()
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["DCC_DATA_ROOT"] = str(DATA_ROOT)
    process = subprocess.Popen(
        build_run_command(dry_run=False),
        cwd=LOCAL_PROJECT_DIR,
        env=child_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            "실험 프로세스가 종료 코드 " + str(return_code) + "로 실패했습니다. "
            "바로 위에 출력된 마지막 Traceback이 실제 원인입니다. 수정 후 모두 실행하면 완료 단계부터 재개합니다."
        )


## 8. 실험 조건과 Training 100% 결과 검증

공식 Validation을 모델 선택에 사용하지 않았는지, 내부 split·hard-negative 대상·sampler·선택 epoch·최종 100% 학습 설정을 모두 검사합니다.


In [ ]:
import json
import math
import torch

if not EXPERIMENT_SUMMARY.is_file():
    raise FileNotFoundError("실험 요약이 없습니다: " + str(EXPERIMENT_SUMMARY))
if not FINAL_CHECKPOINT.is_file():
    raise FileNotFoundError("최종 체크포인트가 없습니다: " + str(FINAL_CHECKPOINT))

experiment = json.loads(EXPERIMENT_SUMMARY.read_text(encoding="utf-8"))
split = experiment["split"]
assert experiment["official_validation_used"] is False
assert split["method"] == "greedy_iterative_multilabel_stratification"
assert math.isclose(float(split["val_ratio"]), INTERNAL_VAL_RATIO)
assert set(experiment["plan"]["hard_negative_symptoms"]) == set(HARD_NEGATIVE_SYMPTOMS)
assert "오심" not in experiment["plan"]["hard_negative_symptoms"]
assert math.isclose(float(experiment["plan"]["hard_negative_oversample"]), 1.5)
assert experiment["plan"]["sampler"] == "guaranteed full pass + hard-negative extras"
assert experiment["selected_variant"] in ("current_input", "speaker_role_tokens")
assert 1 <= int(experiment["selected_epoch"]) <= MAX_EPOCHS
assert len(experiment["variant_results"]) == 2

try:
    checkpoint = torch.load(FINAL_CHECKPOINT, map_location="cpu", weights_only=False, mmap=True)
except (TypeError, RuntimeError):
    checkpoint = torch.load(FINAL_CHECKPOINT, map_location="cpu", weights_only=False)

assert checkpoint["epoch"] == int(experiment["selected_epoch"])
assert checkpoint["model_name"] == MODEL_NAME
assert checkpoint["split_config"]["method"] == "final_training_100_percent"
assert checkpoint["split_config"]["internal_validation"] == 0
assert checkpoint["sampling_config"]["name"] == "guaranteed_full_pass_hard_negative_sampler"
assert checkpoint["sampling_config"]["guarantees_every_sample_each_epoch"] is True
assert checkpoint["sampling_config"]["base_samples"] == checkpoint["split_config"]["num_train"]
assert set(checkpoint["sampling_config"]["focus_symptoms"]) == set(HARD_NEGATIVE_SYMPTOMS)
assert "오심" not in checkpoint["sampling_config"]["focus_symptoms"]

print("선택 모델:", experiment["selected_variant"])
print("선택 epoch:", experiment["selected_epoch"])
print("내부 검증 Macro F1@0.5:", experiment["selected_macro_f1_fixed_0_5"])
print("Training 100% 파일 수:", checkpoint["split_config"]["num_train"])
print("공식 Validation 모델 선택 사용:", experiment["official_validation_used"])
print("실험 조건 검증: PASS")

del checkpoint


## 9. 공식 Validation 최종 평가

모델과 epoch 선택이 끝난 뒤에만 공식 Validation을 한 번 평가합니다. 내부 검증에서 선택한 증상별 임계값을 체크포인트에서 읽어 사용합니다. 이미 결과 파일이 있으면 재평가하지 않습니다.


In [ ]:
if OFFICIAL_METRICS.is_file():
    print("기존 공식 Validation 결과를 재사용합니다:", OFFICIAL_METRICS)
else:
    evaluate_command = [
        sys.executable, str(LOCAL_PROJECT_DIR / "evaluate.py"),
        "--data_root", str(DATA_ROOT),
        "--ckpt_path", str(FINAL_CHECKPOINT),
        "--metrics_output", str(OFFICIAL_METRICS),
        "--batch_size", "1",
        "--threshold", "0.5",
    ]
    if ALLOW_INTERNAL_THRESHOLD_TUNING:
        evaluate_command.append("--use_checkpoint_thresholds")
    subprocess.run(evaluate_command, cwd=LOCAL_PROJECT_DIR, check=True)


## 10. 전체 및 증상별 성능 확인

공식 Validation 결과에서 전체 지표와 각 증상의 정확도·정밀도·재현율·F1·특이도, TP/FP/FN/TN을 계산합니다. 표는 CSV로도 Google Drive에 저장됩니다.

In [ ]:
import pandas as pd

if not OFFICIAL_METRICS.is_file():
    raise FileNotFoundError(
        f"공식 Validation 지표가 없습니다: {OFFICIAL_METRICS}\n"
        "먼저 '공식 Validation 최종 평가' 셀을 실행하세요."
    )

metrics = json.loads(OFFICIAL_METRICS.read_text(encoding="utf-8"))
num_samples = int(metrics["num_samples"])
if num_samples <= 0:
    raise ValueError("공식 Validation 표본 수가 0입니다.")

# 전체(multilabel) 성능
overall_name_map = {
    "num_samples": "검증 표본 수",
    "macro_precision": "Macro 정밀도",
    "macro_recall": "Macro 재현율",
    "macro_f1": "Macro F1",
    "micro_precision": "Micro 정밀도",
    "micro_recall": "Micro 재현율",
    "micro_f1": "Micro F1",
    "weighted_f1": "Weighted F1",
    "subset_accuracy": "완전일치 정확도",
    "label_accuracy": "전체 라벨 정확도",
    "hamming_loss": "Hamming loss",
}
overall_row = {
    korean_name: metrics.get(metric_name)
    for metric_name, korean_name in overall_name_map.items()
}
overall_table = pd.DataFrame([overall_row])
print("[전체 공식 Validation 성능]")
display(overall_table.round(6))

# 증상별 one-vs-rest 성능
required_counts = {"tp", "fp", "fn", "tn"}
symptom_rows = []
for symptom, values in metrics["per_label"].items():
    missing = required_counts.difference(values)
    if missing:
        raise KeyError(f"{symptom}에 혼동행렬 값이 없습니다: {sorted(missing)}")

    tp, fp = int(values["tp"]), int(values["fp"])
    fn, tn = int(values["fn"]), int(values["tn"])
    total = tp + fp + fn + tn
    if total != num_samples:
        raise ValueError(
            f"{symptom}: TP+FP+FN+TN={total}, num_samples={num_samples}"
        )

    accuracy = (tp + tn) / total
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1_score = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    specificity = tn / (tn + fp) if tn + fp else 0.0
    balanced_accuracy = (recall + specificity) / 2

    symptom_rows.append({
        "증상": symptom,
        "정확도": accuracy,
        "균형정확도": balanced_accuracy,
        "정밀도": precision,
        "재현율": recall,
        "F1-score": f1_score,
        "특이도": specificity,
        "실제 양성": tp + fn,
        "예측 양성": tp + fp,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    })

symptom_metrics = pd.DataFrame(symptom_rows)
score_columns = [
    "정확도", "균형정확도", "정밀도", "재현율", "F1-score", "특이도"
]
count_columns = ["실제 양성", "예측 양성", "TP", "FP", "FN", "TN"]

print("[증상별 성능 - 점수는 0~1, 높을수록 좋음]")
symptom_metrics_display = symptom_metrics.copy()
symptom_metrics_display[score_columns] = (
    symptom_metrics_display[score_columns].round(4)
)
display(symptom_metrics_display)

PER_SYMPTOM_CSV = DRIVE_RUN_DIR / "official_validation_per_symptom_metrics.csv"
symptom_metrics.to_csv(PER_SYMPTOM_CSV, index=False, encoding="utf-8-sig")
print("공식 Validation 원본:", OFFICIAL_METRICS)
print("증상별 성능 CSV 저장:", PER_SYMPTOM_CSV)
print("주의: 일반 정확도는 음성(TN)이 많으면 높아질 수 있으므로",
      "정밀도·재현율·F1·균형정확도를 함께 보세요.")

## 11. 최종 완료 판정 및 증빙 저장

실험 요약, 최종 체크포인트, 공식 Validation 결과가 모두 있고 필수 조건이 일치할 때만 `FINAL_VERIFICATION.json`을 저장합니다.


In [ ]:
from datetime import datetime, timezone

required_files = [EXPERIMENT_SUMMARY, FINAL_CHECKPOINT, OFFICIAL_METRICS]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("최종 결과 파일이 없습니다: " + str(missing_files))

experiment = json.loads(EXPERIMENT_SUMMARY.read_text(encoding="utf-8"))
metrics = json.loads(OFFICIAL_METRICS.read_text(encoding="utf-8"))
verification = {
    "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "data_root": str(DATA_ROOT),
    "baseline_checkpoint": str(BASELINE_CHECKPOINT),
    "experiment_summary": str(EXPERIMENT_SUMMARY),
    "final_checkpoint": str(FINAL_CHECKPOINT),
    "official_metrics": str(OFFICIAL_METRICS),
    "selected_variant": experiment["selected_variant"],
    "selected_epoch": int(experiment["selected_epoch"]),
    "internal_macro_f1_fixed_0_5": float(experiment["selected_macro_f1_fixed_0_5"]),
    "official_macro_f1": float(metrics["macro_f1"]),
    "conditions": {
        "multilabel_stratified_90_10": True,
        "nausea_hard_negative_excluded": True,
        "weakness_dyspnea_oversample_1_5": True,
        "guaranteed_full_pass_sampler": True,
        "current_vs_role_token_compared": True,
        "epoch_selected_by_internal_macro_f1": True,
        "thresholds_selected_only_on_internal_validation": ALLOW_INTERNAL_THRESHOLD_TUNING,
        "final_training_100_percent": True,
        "official_validation_used_for_selection": False,
    },
}
pending = FINAL_VERIFICATION.with_suffix(".json.pending")
pending.write_text(json.dumps(verification, ensure_ascii=False, indent=2), encoding="utf-8")
os.replace(pending, FINAL_VERIFICATION)

print("최종 검증: PASS")
print("증빙:", FINAL_VERIFICATION)
print("공식 Validation Macro F1:", verification["official_macro_f1"])


## Checks

정상 완료 기준:

- `실험 조건 검증: PASS`
- 비교 결과가 `current_input`, `speaker_role_tokens` 두 개
- 선택 epoch가 내부 검증의 고정 0.5 Macro F1 기준
- 최종 체크포인트의 split 방식이 `final_training_100_percent`
- sampler가 모든 Training 샘플의 epoch당 1회 노출을 보장
- hard-negative 대상이 전신쇠약·호흡곤란이고 오심은 제외
- `official_validation_used_for_selection: false`
- 마지막에 `최종 검증: PASS`

런타임이 끊기면 설정과 `RUN_NAME`을 바꾸지 말고 **런타임 → 모두 실행**을 다시 누르세요. 새 실험을 시작할 때만 `RUN_NAME`을 새 이름으로 바꾸세요.
